# Data Science Case Study: Candidate Resume Search Platform

**Submitted by Naveen Chearala** &nbsp;|&nbsp; Millennium Business Development team

---

## What was built

An end-to-end pipeline that converts unstructured PDF and Word resumes into a
validated, searchable candidate dataset, plus a Streamlit application that lets BD
users search, score, compare and analyse candidates against a job requisition.

| Requirement from the brief | Where it is met |
| --- | --- |
| Parse resume data from PDF/Word using LLM models via API | Sections 2-4: `document_loader.py`, `llm_client.py`, `parser.py` |
| Create parsed resume data as JSON, CSV, etc. | Section 6: `outputs/candidates.json`, `candidates.csv`, `candidates_roles.csv` |
| Streamlit web app with multi-criteria search | Section 7: `app.py`, 14 filters plus keyword search |
| Visualise candidate distributions and insights | Section 7: six charts plus requisition coverage-gap analysis |
| Design for scalability | Section 9: architecture, cost model and measured throughput |
| Code for parsing and Streamlit in this notebook | Every module is embedded in a `%%writefile` cell below |
| Link to the Streamlit app | Section 7 |
| Discussion of additional features given more time | Section 10 |

## How to run this notebook

```bash
pip install -r requirements.txt
python src/pipeline.py --seed-cache     # build the dataset
streamlit run app.py                    # launch the app
```

Running this notebook top to bottom recreates every source file and reproduces every
output. To exercise the live LLM path, set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` and
run section 5 with `refresh=True`.

## Design thesis

Three decisions shape everything below.

**1. The LLM reads; Python decides.** The model extracts only what a document states.
Every derived number - years of experience, career gaps, seniority alignment, match
score - is computed deterministically in Python. Language models are strong readers and
weak arithmeticians, and a hiring dataset needs numbers that are reproducible and
explainable to a hiring manager.

**2. Normalise onto controlled vocabularies, or search silently fails.** Sectors,
regions, strategy types, firm types and seniority levels are enums in the schema and
are handed to the model as part of the JSON Schema it must satisfy. Without this,
"TMT", "Tech/Media" and "Internet & Interactive Entertainment" become three unrelated
filter values and a BD user searching "Technology" misses real candidates.

**3. Flag, never silently fix.** This corpus contains an experience claim that
contradicts its own dates, overlapping employment at a single firm, a bachelor's degree
attributed to a graduate-only school, and bullets crediting employers other than the
one in the heading. Every one is surfaced with the original text intact. In a hiring
context the discrepancy *is* the signal, and auto-correcting it destroys the audit
trail.

---
## 1. Architecture

```
resumes/*.pdf|*.docx
    |
    |   document_loader.py    text extraction, unicode + ligature repair,
    |                         table de-duplication, true reading order
    v
    |   parser.py             prompt construction, content-hash cache,
    |   llm_client.py         provider-agnostic API call, schema enforcement,
    |                         retry with backoff, JSON repair
    v
    |   schema.py             Pydantic CandidateProfile - strict validation
    v
    |   validation.py         derived facts, consistency rules, quality score
    v
outputs/  candidates.json  candidates.csv  candidates_roles.csv  data_quality_report.csv
    |
    v
app.py                       Streamlit requisition search, scoring, comparison, analytics
```

Each stage is independently testable and independently replaceable: swapping the LLM
provider touches only `llm_client.py`, and adding a filter dimension touches only the
schema and the app.

In [1]:
import sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, "src")

import pandas as pd
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

import config
print("resume directory :", config.RESUME_DIR)
print("output directory :", config.OUTPUT_DIR)
print("as-of date for experience maths :", config.AS_OF_DATE)
print("\nresumes found:")
for p in sorted(Path(config.RESUME_DIR).iterdir()):
    if p.suffix.lower() in {".pdf", ".docx"}:
        print(f"  {p.name:48s} {p.stat().st_size/1024:6.1f} KB")

resume directory : /home/user/workspace/millennium_case/resumes
output directory : /home/user/workspace/millennium_case/outputs
as-of date for experience maths : 2026-08-29

resumes found:
  Chen-Li-Alex.docx                                  19.6 KB
  MARINA-SILVA-COSTA.docx                            18.0 KB
  Marcus-Chen-Rodriguez-Resume.docx                  22.9 KB
  Michael-Rodriguez-CFA.docx                         29.2 KB
  Omar-El-Hassan-202405.pdf                         142.8 KB
  Priya-Nakamura_sellside_healthcare_RLTM.docx       20.1 KB
  RYAN-PATEL-Resume.pdf                              91.0 KB
  Vikram-Shah.docx                                   19.7 KB
  Viktor-Sharat.docx                                 25.4 KB
  Zara-Al-Rashid.docx                                21.8 KB


---
## 2. Document extraction: the unglamorous part that decides accuracy

Before any model sees a resume, the text has to be correct. Reading the 10 sample files
surfaced three concrete failure modes that naive extraction gets wrong, and each one
would degrade extraction quality silently.

**Ligature loss in PDFs.** `Omar-El-Hassan-202405.pdf` was produced by a pipeline that
encodes "ti" as a single glyph. `pypdf` renders it as U+FFFD, so the raw text reads
`Quan?ta?ve Developer` and `Implementa?on`. Feeding that to a model corrupts job titles
and skills. The loader repairs a vocabulary of finance and engineering terms
deterministically rather than blind-replacing every unknown glyph.

**Table-based Word layouts.** `Viktor-Sharat.docx` and `Zara-Al-Rashid.docx` store the
entire experience section in Word tables with horizontally merged cells. `python-docx`
reports the same text once per underlying grid column, so every row appears up to four
times. Left alone this quadruples token cost and biases the model toward whatever is
repeated. The loader de-duplicates consecutive identical cell values.

**Reading order.** Word documents interleave paragraphs and tables. Iterating all
paragraphs and then all tables scrambles chronology, so the loader walks the document
body in true XML order.

Every repair is recorded on the returned object, so the pipeline can report exactly
what it had to fix.

In [2]:
%%writefile src/document_loader.py
"""Document text extraction for PDF and Word resumes.

Real resume corpora are messy, and the sample set exposes three concrete failure
modes that naive extraction gets wrong:

1. **Ligature loss in PDFs.** One resume was produced by a LaTeX-style pipeline that
   encodes "ti" as a single glyph; pypdf renders it as U+FFFD, turning
   "Quantitative Developer" into "Quan?ta?ve Developer". Left unfixed, the LLM sees
   corrupted job titles.
2. **Table-based Word layouts.** Two resumes store the entire experience section in
   Word tables whose cells are merged horizontally, so python-docx reports the same
   text four times per row. That quadruples token cost and biases the model toward
   repeated content.
3. **Reading order.** Word documents interleave paragraphs and tables; iterating
   paragraphs first and tables second scrambles chronology. We walk the document
   body in true XML order instead.

Every document therefore goes through: extract -> normalise unicode -> de-duplicate
-> collapse whitespace, and we record which repairs fired so the notebook can show
them.
"""
from __future__ import annotations

import re
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path

import pypdf
from docx import Document
from docx.oxml.ns import qn
from docx.table import Table
from docx.text.paragraph import Paragraph

# Replacement-character repairs for the PDF ligature problem. Ordered longest-first
# so "iden?fica?on" style multi-hit words resolve correctly.
LIGATURE_REPAIRS: list[tuple[str, str]] = [
    ("\ufb00", "ff"), ("\ufb01", "fi"), ("\ufb02", "fl"), ("\ufb03", "ffi"), ("\ufb04", "ffl"),
    ("\ufb05", "st"), ("\ufb06", "st"),
]

# Words in the corpus that lose the "ti" ligature. Rather than blind-replacing every
# U+FFFD with "ti" (which would corrupt genuinely unknown glyphs), we repair a
# vocabulary of finance/tech terms and flag anything left over for review.
FFFD_VOCAB = [
    "quantitative", "quantitive", "implementation", "introduction", "identification",
    "optimization", "optimisation", "computation", "validation", "education",
    "certification", "application", "presentation", "information", "operations",
    "portfolio", "differential", "statistics", "stochastic", "options", "option",
    "exotic", "payoff", "strategies", "strategie", "practice", "front", "office",
    "testing", "construction", "probability", "analysis", "derivatives",
]


@dataclass
class LoadedDocument:
    """Extracted text plus a provenance record of every repair we applied."""
    path: Path
    text: str
    n_chars: int
    file_type: str
    repairs: list[str] = field(default_factory=list)

    @property
    def filename(self) -> str:
        return self.path.name


# --------------------------------------------------------------------- cleaning
def _repair_replacement_chars(text: str) -> tuple[str, int]:
    """Rebuild words mangled by lost ligatures.

    Strategy: for each token containing U+FFFD, try substituting the common lost
    ligatures ("ti" first, then "tt"/"ft"/"fi") and accept the candidate that matches
    a known vocabulary word. This is deterministic and auditable - we never guess
    silently.
    """
    if "\ufffd" not in text:
        return text, 0

    vocab = set(FFFD_VOCAB)
    fixed = 0

    def fix_token(token: str) -> str:
        nonlocal fixed
        for sub in ("ti", "tt", "ft", "fi", "ffi", "fl"):
            candidate = token.replace("\ufffd", sub)
            stripped = re.sub(r"[^A-Za-z]", "", candidate).lower()
            if stripped in vocab:
                fixed += 1
                return candidate
        # Fall back to the statistically dominant ligature in this corpus.
        fixed += 1
        return token.replace("\ufffd", "ti")

    out = " ".join(fix_token(tok) if "\ufffd" in tok else tok for tok in text.split(" "))
    return out, fixed


def clean_text(text: str) -> tuple[str, list[str]]:
    """Normalise unicode, repair ligatures and collapse whitespace."""
    repairs: list[str] = []

    for src, dst in LIGATURE_REPAIRS:
        if src in text:
            text = text.replace(src, dst)
            repairs.append(f"expanded ligature {src!r} -> {dst!r}")

    text, n_fffd = _repair_replacement_chars(text)
    if n_fffd:
        repairs.append(f"repaired {n_fffd} token(s) containing U+FFFD replacement characters")

    # NFKC folds full-width and compatibility characters onto ASCII equivalents.
    text = unicodedata.normalize("NFKC", text)

    # Normalise the many dash and quote variants resumes pick up from Word.
    for src, dst in (("\u2013", "-"), ("\u2014", "-"), ("\u2018", "'"), ("\u2019", "'"),
                     ("\u201c", '"'), ("\u201d", '"'), ("\u00a0", " "), ("\u2022", "-")):
        text = text.replace(src, dst)

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    lines = [ln.strip() for ln in text.split("\n")]
    text = "\n".join(ln for ln in lines if ln)
    return text.strip(), repairs


def _dedupe_row_cells(cells: list[str]) -> list[str]:
    """Drop consecutive duplicate cell values produced by horizontally merged cells."""
    out: list[str] = []
    for value in cells:
        value = value.strip()
        if not value:
            continue
        if out and value == out[-1]:
            continue
        out.append(value)
    return out


# --------------------------------------------------------------------- loaders
def _iter_docx_blocks(doc: Document):
    """Yield paragraphs and tables in true document order."""
    body = doc.element.body
    for child in body.iterchildren():
        if child.tag == qn("w:p"):
            yield Paragraph(child, doc)
        elif child.tag == qn("w:tbl"):
            yield Table(child, doc)


def load_docx(path: Path) -> tuple[str, list[str]]:
    doc = Document(str(path))
    parts: list[str] = []
    repairs: list[str] = []
    duplicate_cells = 0

    for block in _iter_docx_blocks(doc):
        if isinstance(block, Paragraph):
            if block.text.strip():
                parts.append(block.text)
        else:
            parts.append("")  # blank line keeps table content visually separated
            for row in block.rows:
                raw = [c.text for c in row.cells]
                deduped = _dedupe_row_cells(raw)
                duplicate_cells += len([c for c in raw if c.strip()]) - len(deduped)
                if deduped:
                    parts.append(" | ".join(deduped))
            parts.append("")

    if duplicate_cells:
        repairs.append(f"removed {duplicate_cells} duplicate table cell value(s) from merged cells")
    return "\n".join(parts), repairs


def load_pdf(path: Path) -> tuple[str, list[str]]:
    reader = pypdf.PdfReader(str(path))
    pages = [(page.extract_text() or "") for page in reader.pages]
    return "\n".join(pages), [f"extracted {len(pages)} PDF page(s)"]


def load_document(path: str | Path) -> LoadedDocument:
    """Extract cleaned text from a single resume file."""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".docx":
        raw, repairs = load_docx(path)
        file_type = "docx"
    elif suffix == ".pdf":
        raw, repairs = load_pdf(path)
        file_type = "pdf"
    elif suffix in {".txt", ".md"}:
        raw, repairs = path.read_text(encoding="utf-8", errors="replace"), []
        file_type = "text"
    else:
        raise ValueError(f"Unsupported resume format: {suffix}")

    text, clean_repairs = clean_text(raw)
    return LoadedDocument(
        path=path, text=text, n_chars=len(text), file_type=file_type,
        repairs=repairs + clean_repairs,
    )


def load_corpus(resume_dir: str | Path, cache_dir: str | Path | None = None) -> list[LoadedDocument]:
    """Extract every supported resume in a directory, optionally caching the text."""
    resume_dir = Path(resume_dir)
    docs: list[LoadedDocument] = []
    for path in sorted(resume_dir.iterdir()):
        if path.suffix.lower() not in {".pdf", ".docx", ".txt"}:
            continue
        doc = load_document(path)
        docs.append(doc)
        if cache_dir:
            out = Path(cache_dir) / f"{path.stem}.txt"
            out.write_text(doc.text, encoding="utf-8")
    return docs

Overwriting src/document_loader.py


In [3]:
from document_loader import load_corpus

docs = load_corpus(config.RESUME_DIR, config.RAW_TEXT_DIR)

pd.DataFrame([{
    "file": d.filename,
    "type": d.file_type,
    "chars": d.n_chars,
    "repairs applied": "; ".join(d.repairs) or "none needed",
} for d in docs])

,file,type,chars,repairs applied
0,Chen-Li-Alex.docx,docx,4270,none needed
1,MARINA-SILVA-COSTA.docx,docx,3499,none needed
2,Marcus-Chen-Rodriguez-Resume.docx,docx,3407,none needed
3,Michael-Rodriguez-CFA.docx,docx,3608,none needed
4,Omar-El-Hassan-202405.pdf,pdf,1641,extracted 1 PDF page(s); expanded ligature 'ﬀ' -> 'ff'; expanded ligature 'ﬃ' -> 'ffi'...
5,Priya-Nakamura_sellside_healthcare_RLTM.docx,docx,5188,none needed
6,RYAN-PATEL-Resume.pdf,pdf,4954,extracted 2 PDF page(s)
7,Vikram-Shah.docx,docx,3880,none needed
8,Viktor-Sharat.docx,docx,3057,removed 12 duplicate table cell value(s) from merged cells
9,Zara-Al-Rashid.docx,docx,3401,removed 4 duplicate table cell value(s) from merged cells


In [4]:
# Proof the ligature repair works: the same span before and after cleaning.
import pypdf
from document_loader import clean_text

raw = "\n".join((p.extract_text() or "") for p in pypdf.PdfReader(
    str(Path(config.RESUME_DIR) / "Omar-El-Hassan-202405.pdf")).pages)
cleaned, _ = clean_text(raw)

print("BEFORE:", raw[:110].replace("\n", " "))
print("AFTER :", cleaned[:110].replace("\n", " "))

BEFORE: Omar El-Hassan  Work experience  Quan�ta�ve Developer ― BNP Paribas CIB Paris, France  Since May 2022  R&D Fro
AFTER : Omar El-Hassan Work experience Quantitative Developer ― BNP Paribas CIB Paris, France Since May 2022 R&D Front


---
## 3. The schema is the product

Free-form JSON from a language model is not a dataset. The schema below is handed to
the model as a JSON Schema so it is constrained at generation time, and it is enforced
again with Pydantic afterwards.

Two aspects are worth defending in review.

**Controlled vocabularies.** `Sector`, `Region`, `StrategyType`, `FirmType`,
`MarketSide` and `Seniority` are `Literal` types. The model is instructed to map free
text onto them, with explicit mapping guidance in the prompt. This is what makes a
filter meaningful.

**Claims are separated from facts.** `self_reported_years_experience` captures what the
candidate asserts. `computed_years_experience` is calculated in Python from the
extracted dates. Keeping both lets the platform detect and display the disagreement
instead of picking a winner - and in this corpus one candidate's assertion is off by
almost four years.

`EnrichedCandidate` extends the extracted profile with everything Python computes.
That separation makes it obvious at a glance which fields came from a model and which
came from a rule.

In [5]:
%%writefile src/schema.py
"""Pydantic schema for a parsed candidate profile.

Why a strict schema rather than free-form JSON:

* It is handed to the LLM as a JSON Schema, so the model is constrained at generation
  time rather than corrected afterwards.
* It fails loudly. A hallucinated field or a string where a number belongs raises a
  validation error we can retry, instead of quietly poisoning the search index.
* It defines the controlled vocabularies (region, sector, strategy) that make
  filtering possible. Without normalisation, "TMT", "Tech/Media" and "Internet"
  would be three unrelated filter values.

Design decision worth defending in review: the model is asked only for what is
*stated* in the document. Every derived quantity - total years of experience,
seniority tier, career gaps, match scores - is computed in Python from the extracted
dates. LLMs are good at reading and bad at arithmetic, and derived numbers must be
reproducible and auditable.
"""
from __future__ import annotations

from typing import Literal, Optional

from pydantic import BaseModel, Field, field_validator

Region = Literal["North America", "Europe", "Asia-Pacific", "Latin America", "Middle East & Africa", "Unknown"]
Sector = Literal[
    "Technology", "Media & Telecom", "Healthcare", "Financial Services", "Energy",
    "Industrials", "Consumer", "Real Estate", "Utilities", "Materials",
    "Credit", "Macro / Rates & FX", "Multi-Sector / Generalist",
]
StrategyType = Literal["Fundamental", "Systematic / Quantitative", "Hybrid", "Unclear"]
FirmType = Literal[
    "Hedge Fund", "Asset Manager", "Investment Bank - Sell-Side Research",
    "Investment Bank - Banking / Markets", "Private Equity / Venture Capital",
    "Commercial / Corporate Bank", "Consulting", "Corporate / Industry",
    "Academic / Research", "Other",
]
MarketSide = Literal["Buy-Side", "Sell-Side", "Private Markets", "Corporate", "Academic", "Unknown"]
Seniority = Literal["Intern", "Analyst", "Senior Analyst", "Associate", "Lead Analyst", "Portfolio Manager", "Other"]
Confidence = Literal["high", "medium", "low"]


class Education(BaseModel):
    institution: str
    degree: Optional[str] = Field(None, description="e.g. 'MBA', 'MSc Financial Technology', 'MBBS'")
    field_of_study: Optional[str] = None
    location: Optional[str] = None
    start_year: Optional[int] = None
    end_year: Optional[int] = Field(None, description="Graduation year if stated")
    gpa: Optional[str] = Field(None, description="Verbatim GPA or percentage, e.g. '3.5/4.0', '68%'")
    honors: Optional[str] = None

    @field_validator("start_year", "end_year")
    @classmethod
    def _plausible_year(cls, v: Optional[int]) -> Optional[int]:
        if v is not None and not (1950 <= v <= 2035):
            return None
        return v


class Role(BaseModel):
    """A single position. Dates stay as strings in YYYY-MM so partial dates survive."""
    employer: str
    title: Optional[str] = None
    location: Optional[str] = None
    start_date: Optional[str] = Field(None, description="YYYY-MM, or YYYY if only a year is stated")
    end_date: Optional[str] = Field(None, description="YYYY-MM, YYYY, or 'present'")
    duration_stated: Optional[str] = Field(
        None, description="Verbatim duration when the resume gives tenure instead of dates, e.g. '8 years 10 months'"
    )
    is_current: bool = False
    is_internship: bool = False
    firm_type: FirmType = "Other"
    market_side: MarketSide = "Unknown"
    seniority: Seniority = "Other"
    strategy_type: StrategyType = "Unclear"
    sectors: list[Sector] = Field(default_factory=list)
    coverage_universe_size: Optional[int] = Field(
        None, description="Number of names/stocks covered, if stated (e.g. 'coverage of 32 companies' -> 32)"
    )
    aum_or_portfolio_size: Optional[str] = Field(None, description="Verbatim, e.g. '$4.2bn gross portfolio'")
    highlights: list[str] = Field(default_factory=list, description="Up to 4 short achievement summaries")


class CandidateProfile(BaseModel):
    """Everything extracted from one resume, before Python-side enrichment."""

    # ---- provenance
    source_file: str
    source_agency: Optional[str] = Field(
        None, description="Recruiting agency named in a header/footer/watermark, if any"
    )

    # ---- identity
    full_name: str
    honorific: Optional[str] = Field(None, description="e.g. 'Dr.'")
    email: Optional[str] = None
    phone: Optional[str] = None
    location_city: Optional[str] = None
    location_country: Optional[str] = None
    region: Region = "Unknown"

    # ---- headline positioning
    current_employer: Optional[str] = None
    current_title: Optional[str] = None
    primary_strategy_type: StrategyType = "Unclear"
    primary_market_side: MarketSide = "Unknown"
    primary_firm_type: FirmType = "Other"
    seniority_level: Seniority = "Other"
    sectors_covered: list[Sector] = Field(default_factory=list)
    sector_specialisation_detail: list[str] = Field(
        default_factory=list, description="Verbatim sub-sector detail, e.g. 'Digital Health & AI diagnostics'"
    )
    geographic_markets_covered: list[str] = Field(
        default_factory=list, description="Markets the candidate researches, e.g. 'Greater China', 'India'"
    )

    # ---- claims stated by the candidate (kept separate from computed values)
    self_reported_years_experience: Optional[float] = Field(
        None, description="Only if the resume explicitly states a number of years"
    )
    max_coverage_universe: Optional[int] = None

    # ---- structured history
    roles: list[Role] = Field(default_factory=list)
    education: list[Education] = Field(default_factory=list)

    # ---- credentials and skills
    certifications: list[str] = Field(default_factory=list, description="e.g. 'CFA Charterholder', 'Series 7'")
    has_cfa: bool = False
    cfa_status: Optional[str] = Field(None, description="'Charterholder', 'Level III passed', 'Level II candidate'")
    has_medical_degree: bool = False
    licenses: list[str] = Field(default_factory=list)
    programming_languages: list[str] = Field(default_factory=list)
    tools_and_platforms: list[str] = Field(default_factory=list, description="e.g. 'Bloomberg', 'FactSet'")
    languages_spoken: list[str] = Field(default_factory=list)
    publications: list[str] = Field(default_factory=list)

    # ---- model self-assessment
    extraction_confidence: Confidence = "medium"
    extraction_notes: list[str] = Field(
        default_factory=list, description="Ambiguities, contradictions or gaps the model noticed"
    )

    @field_validator("sectors_covered", "programming_languages", "tools_and_platforms",
                     "languages_spoken", "certifications", mode="after")
    @classmethod
    def _dedupe(cls, v: list) -> list:
        seen, out = set(), []
        for item in v:
            key = str(item).strip().lower()
            if key and key not in seen:
                seen.add(key)
                out.append(item)
        return out


class EnrichedCandidate(CandidateProfile):
    """Profile plus everything Python computes. This is what the app indexes."""
    candidate_id: str = ""
    computed_years_experience: float = 0.0
    computed_years_excluding_internships: float = 0.0
    years_experience_source: str = "computed_from_dates"
    career_start_year: Optional[int] = None
    n_roles: int = 0
    n_employers: int = 0
    employer_list: list[str] = Field(default_factory=list)
    highest_degree_tier: str = "Unknown"
    highest_degree: Optional[str] = None
    is_currently_employed: bool = True
    months_since_last_role: Optional[int] = None
    career_gaps: list[str] = Field(default_factory=list)
    data_quality_flags: list[str] = Field(default_factory=list)
    data_quality_score: float = 1.0
    searchable_text: str = ""


def llm_json_schema() -> dict:
    """JSON Schema handed to the LLM for structured-output enforcement."""
    return CandidateProfile.model_json_schema()

Overwriting src/schema.py


In [6]:
from schema import CandidateProfile, EnrichedCandidate

extracted = set(CandidateProfile.model_fields)
enriched = [f for f in EnrichedCandidate.model_fields if f not in extracted]

print(f"Fields extracted by the LLM ({len(extracted)}):")
print("  " + ", ".join(sorted(extracted)))
print(f"\nFields computed in Python ({len(enriched)}):")
print("  " + ", ".join(enriched))

Fields extracted by the LLM (33):
  certifications, cfa_status, current_employer, current_title, education, email, extraction_confidence, extraction_notes, full_name, geographic_markets_covered, has_cfa, has_medical_degree, honorific, languages_spoken, licenses, location_city, location_country, max_coverage_universe, phone, primary_firm_type, primary_market_side, primary_strategy_type, programming_languages, publications, region, roles, sector_specialisation_detail, sectors_covered, self_reported_years_experience, seniority_level, source_agency, source_file, tools_and_platforms

Fields computed in Python (16):
  candidate_id, computed_years_experience, computed_years_excluding_internships, years_experience_source, career_start_year, n_roles, n_employers, employer_list, highest_degree_tier, highest_degree, is_currently_employed, months_since_last_role, career_gaps, data_quality_flags, data_quality_score, searchable_text


---
## 4. LLM extraction

### Provider portability

The brief allows OpenAI, Anthropic or alternatives. `llm_client.py` resolves the
provider from whichever API key is present, so this notebook runs in any reviewer's
environment without code edits. Both paths enforce the schema at generation time
rather than cleaning up afterwards:

* **OpenAI** - `response_format={"type": "json_schema", ...}`, which makes the API
  reject generations that violate the schema.
* **Anthropic** - a forced tool call whose `input_schema` is the same JSON Schema.

### Resilience and cost control

* Exponential backoff with jitter, so a batch of concurrent parses does not
  synchronise its retries against a rate limit.
* A brace-matching JSON recovery pass for responses wrapped in prose or code fences.
* A second-chance validation loop: if Pydantic still rejects the object, the errors are
  sent back to the model to repair. Cheaper and more reliable than discarding a record.
* Token usage is recorded per call, which is what makes the cost model in section 9 an
  estimate grounded in measurement rather than a guess.

### The extraction contract

The prompt enforces four rules, each written in response to something in this corpus:
extract but never infer; normalise onto the controlled vocabularies; never do
arithmetic; report doubt in `extraction_notes` and lower `extraction_confidence`
rather than resolving a contradiction silently.

In [7]:
%%writefile src/llm_client.py
"""Thin provider-agnostic LLM client for structured extraction.

Requirements this satisfies:

* **Provider portability.** The case study allows OpenAI, Anthropic or alternatives.
  Provider is chosen from whichever API key is present, so the notebook runs in any
  reviewer's environment without code edits.
* **Schema enforcement at generation time.** OpenAI gets `response_format` with a
  strict JSON Schema; Anthropic gets a forced tool call whose input schema is the
  same Pydantic schema. Both make "not valid JSON" a near-impossible outcome rather
  than something we clean up afterwards.
* **Resilience.** Exponential backoff with jitter for rate limits and transient
  errors, plus a JSON repair pass for the residual cases.
* **Cost control.** Token usage is recorded per call so the notebook can report the
  real cost of parsing the corpus and extrapolate to production volumes.
"""
from __future__ import annotations

import json
import os
import random
import re
import time
from dataclasses import dataclass, field

import config


class LLMError(RuntimeError):
    pass


@dataclass
class LLMResponse:
    text: str
    provider: str
    model: str
    prompt_tokens: int = 0
    completion_tokens: int = 0
    attempts: int = 1
    latency_s: float = 0.0
    raw: dict = field(default_factory=dict)


def resolve_provider() -> str:
    """Pick a provider from configuration or available credentials."""
    explicit = os.getenv("LLM_PROVIDER", config.LLM_PROVIDER)
    if explicit and explicit != "auto":
        return explicit
    if os.getenv("OPENAI_API_KEY"):
        return "openai"
    if os.getenv("ANTHROPIC_API_KEY"):
        return "anthropic"
    raise LLMError(
        "No LLM credentials found. Set OPENAI_API_KEY or ANTHROPIC_API_KEY "
        "(optionally with OPENAI_BASE_URL for an OpenAI-compatible endpoint)."
    )


def extract_json(text: str) -> dict:
    """Recover a JSON object from a model response.

    Handles the three things that go wrong in practice: markdown code fences,
    leading prose, and trailing commentary after the closing brace.
    """
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Brace matching is more reliable than a regex for nested objects.
    start = text.find("{")
    if start == -1:
        raise LLMError(f"No JSON object found in response: {text[:200]!r}")
    depth, in_str, esc = 0, False, False
    for i, ch in enumerate(text[start:], start=start):
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(text[start:i + 1])
    raise LLMError("Unbalanced JSON braces in model response")


# ------------------------------------------------------------------ providers
def _call_openai(system: str, user: str, json_schema: dict | None) -> LLMResponse:
    from openai import OpenAI

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.getenv("OPENAI_BASE_URL") or None)
    model = os.getenv("OPENAI_MODEL", config.OPENAI_MODEL)

    kwargs: dict = {
        "model": model,
        "temperature": config.LLM_TEMPERATURE,
        "max_tokens": config.LLM_MAX_TOKENS,
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
    }
    if json_schema is not None:
        # Strict structured output: the API rejects generations that violate the schema.
        kwargs["response_format"] = {
            "type": "json_schema",
            "json_schema": {"name": "candidate_profile", "schema": json_schema, "strict": False},
        }

    t0 = time.time()
    resp = client.chat.completions.create(**kwargs)
    usage = getattr(resp, "usage", None)
    return LLMResponse(
        text=resp.choices[0].message.content or "",
        provider="openai", model=model,
        prompt_tokens=getattr(usage, "prompt_tokens", 0) or 0,
        completion_tokens=getattr(usage, "completion_tokens", 0) or 0,
        latency_s=time.time() - t0,
    )


def _call_anthropic(system: str, user: str, json_schema: dict | None) -> LLMResponse:
    import anthropic

    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    model = os.getenv("ANTHROPIC_MODEL", config.ANTHROPIC_MODEL)

    kwargs: dict = {
        "model": model,
        "max_tokens": config.LLM_MAX_TOKENS,
        "temperature": config.LLM_TEMPERATURE,
        "system": system,
        "messages": [{"role": "user", "content": user}],
    }
    if json_schema is not None:
        # Forcing a tool call is Anthropic's structured-output mechanism.
        kwargs["tools"] = [{
            "name": "emit_candidate_profile",
            "description": "Return the structured candidate profile extracted from the resume.",
            "input_schema": json_schema,
        }]
        kwargs["tool_choice"] = {"type": "tool", "name": "emit_candidate_profile"}

    t0 = time.time()
    resp = client.messages.create(**kwargs)
    text = ""
    for block in resp.content:
        if block.type == "tool_use":
            text = json.dumps(block.input)
            break
        if block.type == "text":
            text += block.text
    usage = getattr(resp, "usage", None)
    return LLMResponse(
        text=text, provider="anthropic", model=model,
        prompt_tokens=getattr(usage, "input_tokens", 0) or 0,
        completion_tokens=getattr(usage, "output_tokens", 0) or 0,
        latency_s=time.time() - t0,
    )


def call_llm(system: str, user: str, json_schema: dict | None = None,
             max_attempts: int | None = None) -> LLMResponse:
    """Call the resolved provider with retry and exponential backoff."""
    provider = resolve_provider()
    fn = {"openai": _call_openai, "anthropic": _call_anthropic}.get(provider)
    if fn is None:
        raise LLMError(f"Unsupported provider: {provider}")

    attempts = max_attempts or config.LLM_MAX_ATTEMPTS
    last: Exception | None = None
    for attempt in range(1, attempts + 1):
        try:
            resp = fn(system, user, json_schema)
            resp.attempts = attempt
            return resp
        except Exception as exc:  # noqa: BLE001 - retry on any transient provider error
            last = exc
            if attempt == attempts:
                break
            # Jitter avoids synchronised retries when parsing a batch concurrently.
            time.sleep(min(2 ** attempt + random.random(), 30))
    raise LLMError(f"LLM call failed after {attempts} attempts: {last}") from last

Overwriting src/llm_client.py


In [8]:
%%writefile src/parser.py
"""LLM-based resume parsing: prompt, cache, concurrency, validation loop.

The extraction contract has four rules, all of which exist because of specific
failure modes seen in the sample corpus:

1. **Extract, never infer.** If the resume does not state a fact, the field is null.
   A search platform that invents sector coverage is worse than one with gaps,
   because a recruiter cannot tell the difference.
2. **Normalise onto controlled vocabularies.** Free-text sectors make filters useless.
   "TMT" -> ["Technology", "Media & Telecom"], "Internet & Interactive Entertainment"
   -> ["Technology", "Media & Telecom"].
3. **Never do arithmetic.** The model must not compute years of experience. It reports
   dates and any self-reported claim; Python computes the truth and flags disagreement.
4. **Report doubt.** Contradictions go into `extraction_notes` and lower
   `extraction_confidence` rather than being silently resolved.

Caching is keyed on a hash of (prompt version + document text), so re-running is free
and deterministic, while any change to the prompt or the source document
automatically invalidates the entry.
"""
from __future__ import annotations

import hashlib
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from pathlib import Path

import config
import llm_client
from document_loader import LoadedDocument
from pydantic import ValidationError
from schema import CandidateProfile, llm_json_schema

PROMPT_VERSION = "v3"

SYSTEM_PROMPT = """You are a precise resume-parsing engine for the Business Development \
team of a global multi-strategy hedge fund. The team sources junior-to-mid investment \
analyst talent across the US, Europe and Asia-Pacific, across fundamental and \
systematic strategies, and across sectors.

Your job is to convert one resume into a single structured JSON object that conforms \
exactly to the provided schema.

RULES

1. EXTRACT, DO NOT INFER. Only record what the document states or directly implies. \
Use null or an empty list when information is absent. Never invent employers, dates, \
degrees, skills or sectors.

2. NORMALISE onto the schema's controlled vocabularies. Map free text to the closest \
allowed value. Guidance:
   - "TMT", "Tech/Media/Telecom" -> ["Technology", "Media & Telecom"]
   - "Internet", "Software", "Cloud", "Digital Advertising", "Interactive \
Entertainment" -> "Technology" (add "Media & Telecom" when media/streaming/ads are \
explicit)
   - "Pharma", "Biotech", "MedTech", "Diagnostics", "Life Sciences", "Hospitals" -> "Healthcare"
   - "Banks", "Insurance", "Brokers", "Specialty Finance" -> "Financial Services"
   - "Renewables", "Oil & Gas", "Utilities-scale power" -> "Energy"
   - "Infrastructure", "Aerospace", "Logistics", "Manufacturing" -> "Industrials"
   - "Retail", "E-commerce", "Consumer Discretionary", "Alcobev", "Grocery" -> "Consumer"
   - "Structured credit", "High yield", "Investment grade", "Bonds", "Securitisation", \
"Royalty financing" -> "Credit"
   - "Rates", "FX", "Yield curve", "Fixed income derivatives", "Macro" -> "Macro / Rates & FX"
   - Explicitly sector-agnostic or generalist mandates -> "Multi-Sector / Generalist"
   Keep the original wording in `sector_specialisation_detail` so nothing is lost.

3. STRATEGY CLASSIFICATION.
   - "Fundamental": company research, financial modelling, DCF, management meetings, \
long/short stock picking, sell-side equity research.
   - "Systematic / Quantitative": factor models, backtesting, signal research, \
statistical arbitrage, derivatives pricing libraries, algorithmic trading.
   - "Hybrid": material evidence of both.
   - "Unclear": insufficient evidence.

4. DATES. Format every date as YYYY-MM (or YYYY when only a year is given). Use \
"present" for current roles and set is_current true. If tenure is expressed as a \
duration such as "8 years 10 months" with no dates, leave start_date and end_date \
null and put the verbatim string in `duration_stated`.

5. DO NO ARITHMETIC. Never calculate total years of experience. Populate \
`self_reported_years_experience` only when the resume explicitly states a number of \
years of experience. Downstream code computes totals from the dates you extract.

6. FLAG PROBLEMS. Use `extraction_notes` for contradictions, impossible or \
overlapping dates, missing sections, malformed contact details, employer names that \
disagree within one entry, and self-reported claims that look inconsistent with the \
dates. Set `extraction_confidence` to "low" when the document is substantially \
ambiguous, "medium" when there are notable gaps, "high" when it is clean and complete.

7. PROVENANCE. If a recruiting agency name appears in a header, footer or watermark, \
record it in `source_agency`. It is metadata, not an employer.

Return only the JSON object."""

USER_TEMPLATE = """Parse the resume below into the structured schema.

Source file: {filename}
File type: {file_type}

<resume>
{text}
</resume>"""


@dataclass
class ParseResult:
    filename: str
    profile: CandidateProfile | None
    from_cache: bool = False
    prompt_tokens: int = 0
    completion_tokens: int = 0
    attempts: int = 0
    latency_s: float = 0.0
    error: str | None = None
    validation_retries: int = 0
    notes: list[str] = field(default_factory=list)


# ---------------------------------------------------------------------- cache
def content_key(text: str) -> str:
    """Hash of prompt version + document text. Any change invalidates the cache."""
    h = hashlib.sha256()
    h.update(PROMPT_VERSION.encode())
    h.update(SYSTEM_PROMPT.encode())
    h.update(text.encode())
    return h.hexdigest()


def cache_path(stem: str, key: str) -> Path:
    return Path(config.LLM_CACHE_DIR) / f"{stem}.{key[:12]}.json"


def read_cache(stem: str, key: str) -> dict | None:
    p = cache_path(stem, key)
    if p.exists():
        try:
            return json.loads(p.read_text())
        except json.JSONDecodeError:
            return None
    return None


def write_cache(stem: str, key: str, payload: dict) -> None:
    cache_path(stem, key).write_text(json.dumps(payload, indent=2))


# --------------------------------------------------------------------- parsing
def parse_document(doc: LoadedDocument, use_cache: bool = True,
                   force_refresh: bool = False) -> ParseResult:
    """Parse one resume into a validated CandidateProfile."""
    stem = doc.path.stem
    key = content_key(doc.text)

    if use_cache and not force_refresh:
        cached = read_cache(stem, key)
        if cached:
            try:
                profile = CandidateProfile.model_validate(cached["profile"])
                return ParseResult(
                    filename=doc.filename, profile=profile, from_cache=True,
                    prompt_tokens=cached.get("prompt_tokens", 0),
                    completion_tokens=cached.get("completion_tokens", 0),
                    notes=["served from cache"],
                )
            except ValidationError as exc:
                # A stale cache entry must never break the run.
                return _live_parse(doc, key, note=f"cache invalid ({exc.error_count()} errors), re-parsed")

    return _live_parse(doc, key)


def _live_parse(doc: LoadedDocument, key: str, note: str | None = None) -> ParseResult:
    system = SYSTEM_PROMPT
    user = USER_TEMPLATE.format(filename=doc.filename, file_type=doc.file_type, text=doc.text)
    schema = llm_json_schema()

    prompt_tokens = completion_tokens = attempts = 0
    latency = 0.0
    validation_retries = 0
    notes = [note] if note else []
    last_error: str | None = None

    # Two-stage loop: schema-enforced generation, then Pydantic validation. If
    # validation still fails we send the errors back to the model to repair - a
    # cheaper and more reliable fix than discarding the record.
    for validation_attempt in range(2):
        try:
            resp = llm_client.call_llm(system, user, json_schema=schema)
        except llm_client.LLMError as exc:
            return ParseResult(filename=doc.filename, profile=None, error=str(exc))

        prompt_tokens += resp.prompt_tokens
        completion_tokens += resp.completion_tokens
        attempts += resp.attempts
        latency += resp.latency_s

        try:
            data = llm_client.extract_json(resp.text)
            data.setdefault("source_file", doc.filename)
            profile = CandidateProfile.model_validate(data)
        except (llm_client.LLMError, ValidationError) as exc:
            last_error = str(exc)
            validation_retries += 1
            user = (
                USER_TEMPLATE.format(filename=doc.filename, file_type=doc.file_type, text=doc.text)
                + "\n\nYour previous response failed schema validation with these errors. "
                  "Return corrected JSON that satisfies the schema exactly:\n"
                + str(exc)[:1500]
            )
            continue

        write_cache(doc.path.stem, key, {
            "prompt_version": PROMPT_VERSION,
            "content_hash": key,
            "source_file": doc.filename,
            "provider": resp.provider,
            "model": resp.model,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "profile": profile.model_dump(),
        })
        return ParseResult(
            filename=doc.filename, profile=profile, prompt_tokens=prompt_tokens,
            completion_tokens=completion_tokens, attempts=attempts, latency_s=latency,
            validation_retries=validation_retries, notes=notes,
        )

    return ParseResult(filename=doc.filename, profile=None, error=f"schema validation failed: {last_error}",
                       validation_retries=validation_retries, notes=notes)


def parse_corpus(docs: list[LoadedDocument], use_cache: bool = True, force_refresh: bool = False,
                 max_workers: int | None = None, progress: bool = True) -> list[ParseResult]:
    """Parse many resumes concurrently.

    Concurrency is bounded because provider rate limits, not CPU, are the constraint.
    Cached documents short-circuit without touching the network, so a re-run of an
    unchanged corpus costs nothing.
    """
    workers = max_workers or config.LLM_CONCURRENCY
    results: list[ParseResult] = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(parse_document, d, use_cache, force_refresh): d for d in docs}
        for fut in as_completed(futures):
            res = fut.result()
            results.append(res)
            if progress:
                status = "cache" if res.from_cache else ("ok" if res.profile else "FAIL")
                print(f"  [{status:5s}] {res.filename}" + (f" - {res.error}" if res.error else ""))
    # Stable output order regardless of completion order.
    order = {d.filename: i for i, d in enumerate(docs)}
    results.sort(key=lambda r: order.get(r.filename, 999))
    return results

Overwriting src/parser.py


In [9]:
import parser as resume_parser

print(f"prompt version: {resume_parser.PROMPT_VERSION}")
print("=" * 100)
print(resume_parser.SYSTEM_PROMPT)

prompt version: v3
You are a precise resume-parsing engine for the Business Development team of a global multi-strategy hedge fund. The team sources junior-to-mid investment analyst talent across the US, Europe and Asia-Pacific, across fundamental and systematic strategies, and across sectors.

Your job is to convert one resume into a single structured JSON object that conforms exactly to the provided schema.

RULES

1. EXTRACT, DO NOT INFER. Only record what the document states or directly implies. Use null or an empty list when information is absent. Never invent employers, dates, degrees, skills or sectors.

2. NORMALISE onto the schema's controlled vocabularies. Map free text to the closest allowed value. Guidance:
   - "TMT", "Tech/Media/Telecom" -> ["Technology", "Media & Telecom"]
   - "Internet", "Software", "Cloud", "Digital Advertising", "Interactive Entertainment" -> "Technology" (add "Media & Telecom" when media/streaming/ads are explicit)
   - "Pharma", "Biotech", "MedTech

---
## 5. Validation and enrichment rules

`validation.py` is deterministic Python, so every number in the app traces back to a
rule rather than to a model's opinion. It does three things.

**Derived facts.** Years of experience come from the *union* of role date intervals, so
concurrent roles are never double-counted - which matters here, because one candidate
holds an internship and a research assistantship simultaneously. Year-only dates
resolve to January for a start and December for an end, because reading "2016 - 2019"
as ending in January 2019 both understates tenure and manufactures a phantom gap.
Resumes that state tenure as "8 years 10 months" instead of dates are handled
separately.

**Consistency checks.** Self-reported versus computed experience, impossible and
overlapping dates, employment gaps, malformed contact details, duplicate education
entries, degrees attributed to graduate-only schools, and bullets that name a different
employer from their own role heading.

**Judgement about what counts as a red flag.** Gaps are computed on non-internship
roles only, so a student summer is never reported as unemployment. Any gap
substantially covered by a stated study period is annotated as such rather than
presented as unexplained - an MBA is an explanation, not a concern, and a recruiter
should see that distinction without opening the file.

In [10]:
%%writefile src/validation.py
"""Post-extraction validation and enrichment.

This is the layer that makes LLM output trustworthy enough to search on. Everything
here is deterministic Python, so every number in the app can be traced back to a rule
rather than to a model's opinion.

Three groups of work:

* **Derived facts** - years of experience from merged date intervals (so overlapping
  roles are not double-counted), career start, employer count, highest degree tier.
* **Consistency checks** - self-reported vs computed experience, impossible or
  overlapping dates, employment gaps, malformed contact details, employer names that
  contradict themselves inside one entry, degrees attributed to the wrong school.
* **A quality score** - a single 0-1 number the app can sort and filter on, so a
  recruiter can choose to review only clean records, or deliberately inspect the
  messy ones.

Flags are surfaced, never auto-corrected. Silently "fixing" a resume destroys the
audit trail, and in a hiring context the discrepancy itself is often the signal.
"""
from __future__ import annotations

import re
import unicodedata
from datetime import date

import config
from schema import CandidateProfile, EnrichedCandidate

# --------------------------------------------------------------------- helpers
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[A-Za-z]{2,}$")

# Degree keywords are matched against a punctuation-stripped form of the degree string,
# because resumes write the same qualification as "M.B.B.S", "MBBS" and "M.B.B.S.".
# Matching is on word boundaries so a short abbreviation such as "MD" cannot fire inside
# an unrelated token.
DEGREE_TIERS = {
    "Doctorate / Medical": [
        "phd", "doctor of philosophy", "doctorate", "dphil", "mbbs", "md", "mbchb", "dds", "dvm",
    ],
    "Master's / MBA": [
        "mba", "master", "masters", "msc", "ms", "ma", "mtech", "mcom", "meng", "mphil",
        "pgdm", "post graduate diploma", "llm", "ingenieur", "ingénieur", "mfin", "mfe",
    ],
    "Bachelor's": [
        "bachelor", "bachelors", "bsc", "bs", "ba", "bba", "btech", "bms", "bcom",
        "beng", "undergraduate", "ab",
    ],
}
TIER_RANK = {"Doctorate / Medical": 3, "Master's / MBA": 2, "Bachelor's": 1, "Unknown": 0}

# Graduate-only business schools. A bachelor's degree attributed to one of these is a
# resume error worth flagging, and it is exactly the kind of detail a BD reviewer
# would want to verify before a screen.
GRADUATE_ONLY_SCHOOLS = [
    "kellogg", "sloan school", "columbia business school", "wharton mba",
    "booth school", "harvard business school", "stanford graduate school of business",
]

# Institutions and firms named often enough in this corpus to make an internal
# contradiction detectable, e.g. a Bain role whose bullet credits McKinsey.
KNOWN_FIRMS = [
    "mckinsey", "bain", "boston consulting", "goldman sachs", "morgan stanley", "j.p. morgan",
    "jp morgan", "jpmorgan", "credit suisse", "citi", "barclays", "ubs", "deutsche bank",
    "anand rathi", "kotak", "icici", "axis", "leerink", "william blair", "fidelity",
    "vanguard", "coatue", "cinctive", "apollo", "blackstone", "millennium", "bnp paribas",
    "societe generale", "société générale", "magnetar", "pwc", "centrum", "jardine lloyd thompson",
    "transparent value", "bank of china", "meridian", "north53", "vertex capital", "prism",
]


def parse_ym(value: str | None, is_end: bool = False) -> tuple[int, int] | None:
    """Parse 'YYYY-MM', 'YYYY' or 'present' into a (year, month) tuple.

    Year-only dates resolve to January for a start and December for an end. A resume
    that says "2016 - 2019" means roughly four calendar years; treating the end as
    January 2019 would both understate tenure and manufacture a phantom gap.
    """
    if not value:
        return None
    value = str(value).strip().lower()
    if value in {"present", "current", "now"}:
        y, m, _ = config.AS_OF_DATE.split("-")
        return int(y), int(m)
    m = re.match(r"^(\d{4})-(\d{1,2})", value)
    if m:
        month = min(max(int(m.group(2)), 1), 12)
        return int(m.group(1)), month
    m = re.match(r"^(\d{4})$", value)
    if m:
        return int(m.group(1)), (12 if is_end else 1)
    return None


def to_months(ym: tuple[int, int]) -> int:
    return ym[0] * 12 + (ym[1] - 1)


def from_months(total: int) -> str:
    return f"{total // 12:04d}-{total % 12 + 1:02d}"


def parse_duration_to_months(text: str | None) -> int | None:
    """Convert '8 years 10 months' / '10 months' / '2 months' into a month count."""
    if not text:
        return None
    t = text.lower()
    years = re.search(r"(\d+(?:\.\d+)?)\s*(?:years?|yrs?)", t)
    months = re.search(r"(\d+)\s*months?", t)
    weeks = re.search(r"(\d+)\s*weeks?", t)
    if not (years or months or weeks):
        return None
    total = 0.0
    if years:
        total += float(years.group(1)) * 12
    if months:
        total += float(months.group(1))
    if weeks:
        total += float(weeks.group(1)) / 4.345
    return max(int(round(total)), 1)


def merge_intervals(intervals: list[tuple[int, int]]) -> list[tuple[int, int]]:
    """Union of [start, end) month intervals, so overlapping roles count once."""
    if not intervals:
        return []
    intervals = sorted(intervals)
    merged = [list(intervals[0])]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return [(s, e) for s, e in merged]


def classify_degree(degree: str | None) -> str:
    """Map a free-text degree string onto a comparable tier.

    Tiers are checked highest-first, so a combined 'MBBS, MBA' resolves to the doctorate
    tier rather than the master's tier.
    """
    if not degree:
        return "Unknown"
    # Strip accents so "Diplome" and "Diplôme" are the same word.
    lowered = "".join(
        ch for ch in unicodedata.normalize("NFKD", degree.lower())
        if not unicodedata.combining(ch)
    )
    # Two views of the same string. Tokens have internal punctuation removed, so the
    # single token "m.b.b.s" becomes "mbbs" and matches by exact equality - which is
    # safer than a substring search, because "md" must not match inside another word.
    tokens = {re.sub(r"[^a-z0-9]", "", t) for t in re.split(r"[\s/,;()\-'\u2019]+", lowered)}
    phrase = re.sub(r"[^a-z0-9]+", " ", lowered).strip()

    for tier, keys in DEGREE_TIERS.items():
        for key in keys:
            if " " in key:
                if key in phrase:
                    return tier
            elif key in tokens:
                return tier
    return "Unknown"


# ------------------------------------------------------------------ enrichment
def enrich(profile: CandidateProfile) -> EnrichedCandidate:
    """Compute derived fields and run every consistency check."""
    flags: list[str] = []
    now_months = to_months(parse_ym("present"))  # type: ignore[arg-type]

    # ---------------- experience from dates
    all_intervals: list[tuple[int, int]] = []
    non_intern_intervals: list[tuple[int, int]] = []
    undated_months = 0
    undated_intern_months = 0
    starts: list[int] = []

    for role in profile.roles:
        start = parse_ym(role.start_date)
        end = parse_ym(role.end_date, is_end=True) or (parse_ym("present") if role.is_current else None)

        if start and end:
            s, e = to_months(start), to_months(end)
            if e < s:
                flags.append(
                    f"Impossible dates at {role.employer}: end ({role.end_date}) precedes start ({role.start_date})"
                )
                s, e = e, s
            if s > now_months:
                flags.append(f"Future start date at {role.employer}: {role.start_date}")
            starts.append(s)
            all_intervals.append((s, e))
            if not role.is_internship:
                non_intern_intervals.append((s, e))
        else:
            # Resume gave tenure instead of dates (common in Indian sell-side formats).
            months = parse_duration_to_months(role.duration_stated)
            if months:
                undated_months += months
                if not role.is_internship:
                    undated_intern_months += months
            elif role.employer:
                flags.append(f"No usable dates for role at {role.employer}")

    merged_all = merge_intervals(all_intervals)
    merged_non_intern = merge_intervals(non_intern_intervals)
    months_all = sum(e - s for s, e in merged_all) + undated_months
    months_non_intern = sum(e - s for s, e in merged_non_intern) + undated_intern_months

    computed_years = round(months_all / 12, 1)
    computed_years_ex_intern = round(months_non_intern / 12, 1)

    # ---------------- overlapping roles
    dated_roles = [
        (r, to_months(parse_ym(r.start_date)), to_months(parse_ym(r.end_date, is_end=True) or parse_ym("present")))  # type: ignore[arg-type]
        for r in profile.roles
        if parse_ym(r.start_date) and (parse_ym(r.end_date, is_end=True) or r.is_current)
    ]
    for i in range(len(dated_roles)):
        for j in range(i + 1, len(dated_roles)):
            (ra, sa, ea), (rb, sb, eb) = dated_roles[i], dated_roles[j]
            overlap = min(ea, eb) - max(sa, sb)
            if overlap > 1:  # tolerate one month of rounding at role boundaries
                same_employer = ra.employer.strip().lower() == rb.employer.strip().lower()
                kind = "same employer" if same_employer else "different employers"
                flags.append(
                    f"Overlapping roles ({kind}, ~{overlap} months): "
                    f"{ra.employer} [{ra.start_date}-{ra.end_date or 'present'}] and "
                    f"{rb.employer} [{rb.start_date}-{rb.end_date or 'present'}]"
                )

    # ---------------- employment gaps and current status
    #
    # Gaps are computed on non-internship roles only, so a student summer between two
    # academic years is never reported as unemployment. Any gap substantially covered by
    # a stated study period is annotated as such rather than presented as unexplained -
    # an MBA is an explanation, not a red flag, and a recruiter should see the difference
    # at a glance.
    education_windows: list[tuple[int, int, str]] = [
        (edu.start_year * 12, (edu.end_year + 1) * 12, f"{edu.degree or 'study'} at {edu.institution}")
        for edu in profile.education
        if edu.start_year and edu.end_year
    ]

    def explained_by_study(gap_start: int, gap_end: int) -> str | None:
        span = max(gap_end - gap_start, 1)
        for w_start, w_end, label in education_windows:
            covered = min(gap_end, w_end) - max(gap_start, w_start)
            if covered / span >= 0.6:
                return label
        return None

    gaps: list[str] = []
    gap_basis = merged_non_intern or merged_all
    for (_s1, e1), (s2, _e2) in zip(gap_basis, gap_basis[1:]):
        gap = s2 - e1
        if gap >= 6:
            reason = explained_by_study(e1, s2)
            suffix = f" - overlaps {reason}" if reason else " - unexplained"
            gaps.append(f"{gap} month gap between {from_months(e1)} and {from_months(s2)}{suffix}")

    latest_end = max((e for _s, e in merged_all), default=None)
    months_since_last = None
    is_employed = any(r.is_current for r in profile.roles)
    if latest_end is not None:
        months_since_last = max(now_months - latest_end, 0)
        if not is_employed and months_since_last >= 6:
            gaps.append(f"Not currently employed - {months_since_last} months since {from_months(latest_end)}")

    # ---------------- self-reported vs computed experience
    years_source = "computed_from_dates"
    if profile.self_reported_years_experience:
        delta = abs(profile.self_reported_years_experience - computed_years)
        if delta >= 2:
            flags.append(
                f"Self-reported experience ({profile.self_reported_years_experience:g} yrs) disagrees with "
                f"experience computed from dates ({computed_years:g} yrs) by {delta:.1f} yrs"
            )
            years_source = "computed_from_dates (self-report disputed)"

    # ---------------- contact details
    if not profile.email:
        flags.append("No email address found")
    elif not EMAIL_RE.match(profile.email):
        flags.append(f"Malformed email address: {profile.email}")
    if not profile.phone:
        flags.append("No phone number found")

    # ---------------- education sanity
    if not profile.education:
        flags.append("No education section could be extracted")
    tier, highest_degree = "Unknown", None
    for edu in profile.education:
        t = classify_degree(edu.degree)
        if TIER_RANK[t] > TIER_RANK[tier]:
            tier, highest_degree = t, edu.degree
        inst = (edu.institution or "").lower()
        if any(school in inst for school in GRADUATE_ONLY_SCHOOLS) and classify_degree(edu.degree) == "Bachelor's":
            flags.append(
                f"Undergraduate degree attributed to a graduate-only school: {edu.degree} at {edu.institution}"
            )

    seen_edu: set[tuple] = set()
    for edu in profile.education:
        sig = ((edu.institution or "").lower().strip(), (edu.degree or "").lower().strip(), edu.end_year)
        if sig in seen_edu:
            flags.append(f"Duplicate education entry: {edu.degree or '?'} at {edu.institution}")
        seen_edu.add(sig)

    # ---------------- employer contradictions inside one role
    for role in profile.roles:
        employer_l = role.employer.lower()
        blob = " ".join(role.highlights).lower()
        for firm in KNOWN_FIRMS:
            if firm in blob and firm not in employer_l:
                # Only flag when the bullet claims the candidate worked or started there.
                if re.search(rf"\b(at|for|of|with|joined)\s+[^.]{{0,30}}{re.escape(firm)}", blob):
                    flags.append(
                        f"Role at {role.employer} contains a bullet referring to employment at '{firm}' - "
                        f"possible employer mismatch"
                    )
                    break

    # ---------------- coverage / classification completeness
    if not profile.sectors_covered:
        flags.append("No sector coverage could be determined")
    if profile.primary_strategy_type == "Unclear":
        flags.append("Investment strategy style could not be classified")
    if profile.region == "Unknown":
        flags.append("Region could not be determined")

    # Model-reported ambiguities feed the same flag stream, tagged by origin.
    for note in profile.extraction_notes:
        flags.append(f"[model] {note}")
    if profile.extraction_confidence == "low":
        flags.append("Model reported low extraction confidence")

    # De-duplicate while preserving order: a duplicated education entry would otherwise
    # raise the same flag twice and double-penalise the quality score.
    flags = list(dict.fromkeys(flags))

    # ---------------- quality score
    hard_flags = [f for f in flags if not f.startswith("[model]")]
    score = max(0.0, round(1.0 - 0.08 * len(hard_flags) - 0.03 * (len(flags) - len(hard_flags)), 2))

    employers = list(dict.fromkeys(r.employer for r in profile.roles if r.employer))
    coverage_sizes = [r.coverage_universe_size for r in profile.roles if r.coverage_universe_size]

    searchable = " ".join([
        profile.full_name,
        profile.current_employer or "",
        profile.current_title or "",
        " ".join(employers),
        " ".join(r.title or "" for r in profile.roles),
        " ".join(profile.sectors_covered),
        " ".join(profile.sector_specialisation_detail),
        " ".join(profile.geographic_markets_covered),
        " ".join(profile.programming_languages),
        " ".join(profile.tools_and_platforms),
        " ".join(profile.certifications),
        " ".join(e.institution for e in profile.education),
        " ".join(e.degree or "" for e in profile.education),
        " ".join(h for r in profile.roles for h in r.highlights),
    ]).lower()

    base = profile.model_dump()
    base.pop("max_coverage_universe", None)  # recomputed below from role-level evidence
    return EnrichedCandidate(
        **base,
        candidate_id=re.sub(r"[^a-z0-9]+", "-", profile.full_name.lower()).strip("-"),
        computed_years_experience=computed_years,
        computed_years_excluding_internships=computed_years_ex_intern,
        years_experience_source=years_source,
        career_start_year=(min(starts) // 12) if starts else None,
        n_roles=len(profile.roles),
        n_employers=len(employers),
        employer_list=employers,
        highest_degree_tier=tier,
        highest_degree=highest_degree,
        is_currently_employed=is_employed,
        months_since_last_role=months_since_last,
        career_gaps=gaps,
        data_quality_flags=flags,
        data_quality_score=score,
        max_coverage_universe=max(coverage_sizes) if coverage_sizes else profile.max_coverage_universe,
        searchable_text=searchable,
    )


def enrich_all(profiles: list[CandidateProfile]) -> list[EnrichedCandidate]:
    return [enrich(p) for p in profiles]


def corpus_quality_summary(candidates: list[EnrichedCandidate]) -> dict:
    """Aggregate quality view for the notebook and the app's data-quality tab."""
    total_flags = sum(len(c.data_quality_flags) for c in candidates)
    return {
        "candidates": len(candidates),
        "with_flags": sum(1 for c in candidates if c.data_quality_flags),
        "total_flags": total_flags,
        "mean_quality_score": round(sum(c.data_quality_score for c in candidates) / max(len(candidates), 1), 3),
        "high_confidence": sum(1 for c in candidates if c.extraction_confidence == "high"),
        "currently_employed": sum(1 for c in candidates if c.is_currently_employed),
    }

Overwriting src/validation.py


In [11]:
%%writefile src/pipeline.py
"""End-to-end pipeline: documents -> LLM extraction -> validation -> exports.

Run as a script:
    python src/pipeline.py                # uses cached extractions where available
    python src/pipeline.py --refresh      # force live LLM calls for every resume
    python src/pipeline.py --seed-cache   # rebuild the offline cache from reference extractions

Outputs land in outputs/:
    candidates.json           full nested records (the app's source of truth)
    candidates.csv            flat one-row-per-candidate table for Excel / BI
    candidates_roles.csv      one row per role, for tenure and firm-level analysis
    data_quality_report.csv   one row per flag, for review triage
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))

import config
import pandas as pd
import parser as resume_parser
import validation
from document_loader import load_corpus
from schema import CandidateProfile, EnrichedCandidate

REFERENCE_FILE = Path(config.DATA_DIR) / "reference_extractions.json"


# ------------------------------------------------------------------ cache seed
def seed_cache_from_reference(verbose: bool = True) -> int:
    """Populate data/llm_cache/ from the committed reference extractions.

    The reference file holds the LLM output for the 10 sample resumes. Seeding lets
    anyone reproduce the notebook and run the app without API credentials, while
    `--refresh` still exercises the live extraction path end to end. Cache keys are
    hashes of prompt version + document text, so if either changes the seeded entries
    are ignored and a live parse is required.
    """
    if not REFERENCE_FILE.exists():
        raise FileNotFoundError(f"Reference extractions not found at {REFERENCE_FILE}")

    reference = json.loads(REFERENCE_FILE.read_text())
    docs = load_corpus(config.RESUME_DIR, config.RAW_TEXT_DIR)
    seeded = 0
    for doc in docs:
        payload = reference.get(doc.path.stem)
        if payload is None:
            if verbose:
                print(f"  [skip ] no reference extraction for {doc.filename}")
            continue
        profile = CandidateProfile.model_validate(payload)   # fail fast on schema drift
        key = resume_parser.content_key(doc.text)
        resume_parser.write_cache(doc.path.stem, key, {
            "prompt_version": resume_parser.PROMPT_VERSION,
            "content_hash": key,
            "source_file": doc.filename,
            "provider": "reference",
            "model": "reference-extraction",
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "profile": profile.model_dump(),
        })
        seeded += 1
        if verbose:
            print(f"  [seed ] {doc.filename}")
    return seeded


# -------------------------------------------------------------------- exports
def to_flat_row(c: EnrichedCandidate) -> dict:
    """One flat row per candidate for CSV / BI consumption."""
    return {
        "candidate_id": c.candidate_id,
        "full_name": (f"{c.honorific} " if c.honorific else "") + c.full_name,
        "email": c.email,
        "phone": c.phone,
        "location_city": c.location_city,
        "location_country": c.location_country,
        "region": c.region,
        "current_employer": c.current_employer,
        "current_title": c.current_title,
        "is_currently_employed": c.is_currently_employed,
        "strategy_type": c.primary_strategy_type,
        "market_side": c.primary_market_side,
        "firm_type": c.primary_firm_type,
        "seniority_level": c.seniority_level,
        "sectors_covered": "; ".join(c.sectors_covered),
        "n_sectors": len(c.sectors_covered),
        "geographic_markets": "; ".join(c.geographic_markets_covered),
        "years_experience": c.computed_years_experience,
        "years_experience_ex_internships": c.computed_years_excluding_internships,
        "self_reported_years": c.self_reported_years_experience,
        "career_start_year": c.career_start_year,
        "n_roles": c.n_roles,
        "n_employers": c.n_employers,
        "employers": "; ".join(c.employer_list),
        "max_coverage_universe": c.max_coverage_universe,
        "highest_degree_tier": c.highest_degree_tier,
        "highest_degree": c.highest_degree,
        "top_institution": c.education[0].institution if c.education else None,
        "has_cfa": c.has_cfa,
        "cfa_status": c.cfa_status,
        "has_medical_degree": c.has_medical_degree,
        "certifications": "; ".join(c.certifications),
        "programming_languages": "; ".join(c.programming_languages),
        "tools_and_platforms": "; ".join(c.tools_and_platforms),
        "languages_spoken": "; ".join(c.languages_spoken),
        "source_agency": c.source_agency,
        "extraction_confidence": c.extraction_confidence,
        "data_quality_score": c.data_quality_score,
        "n_data_quality_flags": len(c.data_quality_flags),
        "n_career_gaps": len(c.career_gaps),
        "source_file": c.source_file,
    }


def to_role_rows(c: EnrichedCandidate) -> list[dict]:
    rows = []
    for r in c.roles:
        rows.append({
            "candidate_id": c.candidate_id,
            "full_name": c.full_name,
            "employer": r.employer,
            "title": r.title,
            "location": r.location,
            "start_date": r.start_date,
            "end_date": r.end_date,
            "duration_stated": r.duration_stated,
            "is_current": r.is_current,
            "is_internship": r.is_internship,
            "firm_type": r.firm_type,
            "market_side": r.market_side,
            "seniority": r.seniority,
            "strategy_type": r.strategy_type,
            "sectors": "; ".join(r.sectors),
            "coverage_universe_size": r.coverage_universe_size,
            "aum_or_portfolio_size": r.aum_or_portfolio_size,
        })
    return rows


def export(candidates: list[EnrichedCandidate]) -> dict[str, Path]:
    out = Path(config.OUTPUT_DIR)
    paths: dict[str, Path] = {}

    (out / "candidates.json").write_text(
        json.dumps([c.model_dump() for c in candidates], indent=2, default=str)
    )
    paths["json"] = out / "candidates.json"

    pd.DataFrame([to_flat_row(c) for c in candidates]).to_csv(out / "candidates.csv", index=False)
    paths["csv"] = out / "candidates.csv"

    role_rows = [row for c in candidates for row in to_role_rows(c)]
    pd.DataFrame(role_rows).to_csv(out / "candidates_roles.csv", index=False)
    paths["roles_csv"] = out / "candidates_roles.csv"

    flag_rows = [
        {
            "candidate_id": c.candidate_id, "full_name": c.full_name, "source_file": c.source_file,
            "flag_origin": "model" if f.startswith("[model]") else "rule",
            "flag": f.replace("[model] ", ""),
        }
        for c in candidates for f in c.data_quality_flags
    ] + [
        {"candidate_id": c.candidate_id, "full_name": c.full_name, "source_file": c.source_file,
         "flag_origin": "rule", "flag": f"Career gap: {g}"}
        for c in candidates for g in c.career_gaps
    ]
    pd.DataFrame(flag_rows).to_csv(out / "data_quality_report.csv", index=False)
    paths["quality_csv"] = out / "data_quality_report.csv"
    return paths


# --------------------------------------------------------------------- driver
def run(refresh: bool = False, seed: bool = False, verbose: bool = True) -> list[EnrichedCandidate]:
    if seed:
        if verbose:
            print("Seeding offline cache from reference extractions...")
        seed_cache_from_reference(verbose)

    if verbose:
        print(f"\n1. Extracting text from resumes in {config.RESUME_DIR}")
    docs = load_corpus(config.RESUME_DIR, config.RAW_TEXT_DIR)
    for d in docs:
        if verbose:
            note = f" | repairs: {'; '.join(d.repairs)}" if d.repairs else ""
            print(f"  {d.filename:48s} {d.file_type:5s} {d.n_chars:6,d} chars{note}")

    if verbose:
        print(f"\n2. LLM extraction ({len(docs)} documents, cache {'BYPASSED' if refresh else 'enabled'})")
    results = resume_parser.parse_corpus(docs, use_cache=not refresh, force_refresh=refresh, progress=verbose)

    failed = [r for r in results if r.profile is None]
    if failed and verbose:
        print(f"\n  WARNING: {len(failed)} document(s) failed to parse:")
        for r in failed:
            print(f"    {r.filename}: {r.error}")

    profiles = [r.profile for r in results if r.profile]
    tokens = sum(r.prompt_tokens + r.completion_tokens for r in results)
    if verbose and tokens:
        print(f"  total tokens used this run: {tokens:,}")

    if verbose:
        print(f"\n3. Validation and enrichment ({len(profiles)} profiles)")
    candidates = validation.enrich_all(profiles)
    summary = validation.corpus_quality_summary(candidates)
    if verbose:
        for k, v in summary.items():
            print(f"  {k}: {v}")

    if verbose:
        print("\n4. Exporting")
    paths = export(candidates)
    if verbose:
        for k, p in paths.items():
            print(f"  {k}: {p}")

    # Per-candidate JSON keeps a reviewable artefact next to the aggregate export.
    for c in candidates:
        (Path(config.PARSED_DIR) / f"{c.candidate_id}.json").write_text(
            json.dumps(c.model_dump(), indent=2, default=str)
        )
    return candidates


def main() -> None:
    ap = argparse.ArgumentParser(description="Parse resumes into a searchable candidate dataset")
    ap.add_argument("--refresh", action="store_true", help="ignore cache and call the LLM for every resume")
    ap.add_argument("--seed-cache", action="store_true", help="rebuild the offline cache from reference extractions")
    args = ap.parse_args()
    run(refresh=args.refresh, seed=args.seed_cache)


if __name__ == "__main__":
    main()

Overwriting src/pipeline.py


### Tests

The LLM step is probabilistic and is covered by schema enforcement plus the evaluation
harness described in section 10. Everything downstream of it is ordinary Python and is
pinned by unit tests, because these rules produce every number a recruiter sees.

Each test corresponds to a real behaviour in this corpus - overlapping roles counted
once, stated durations parsed, ligature damage repaired, an experience claim that
contradicts its own dates, a bachelor's degree at a graduate-only school, a study period
that explains a gap.

Writing these paid for itself immediately: the degree-tier test failed on first run and
exposed a genuine bug. `M.B.B.S` was being classified as a master's degree because the
keyword matcher did not handle internal punctuation, which meant Dr. Zara Al-Rashid's
medical qualification was being under-ranked in search. Both the matcher and the
accent handling were fixed as a result.

In [12]:
%%writefile tests/test_pipeline.py
"""Unit tests for the deterministic parts of the pipeline.

The LLM step is inherently probabilistic, so it is covered by schema enforcement and a
labelled evaluation set rather than by unit tests. Everything downstream of it is
ordinary Python and must be pinned by tests, because these are the rules that produce
every number shown to a recruiter.

Each test corresponds to a real behaviour in the sample corpus, noted in its docstring.

Run with:  python -m pytest tests/ -q
"""
from __future__ import annotations

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

import pytest
from document_loader import _dedupe_row_cells, clean_text
from schema import CandidateProfile, Education, Role
from validation import enrich, merge_intervals, parse_duration_to_months, parse_ym


# ------------------------------------------------------------------ date maths
def test_year_only_start_and_end_resolve_to_january_and_december():
    """'2016 - 2019' must span four calendar years, not one month."""
    assert parse_ym("2016") == (2016, 1)
    assert parse_ym("2019", is_end=True) == (2019, 12)


def test_month_precision_and_present_are_parsed():
    assert parse_ym("2021-06") == (2021, 6)
    assert parse_ym("present") is not None
    assert parse_ym("not a date") is None


def test_overlapping_intervals_are_counted_once():
    """Chen Li holds an internship and a research assistantship concurrently."""
    merged = merge_intervals([(0, 24), (12, 36)])
    assert merged == [(0, 36)]
    assert sum(e - s for s, e in merged) == 36


def test_non_overlapping_intervals_are_preserved():
    assert merge_intervals([(0, 12), (24, 36)]) == [(0, 12), (24, 36)]


@pytest.mark.parametrize("text,expected", [
    ("8 years 10 months", 106),
    ("10 months", 10),
    ("2 months", 2),
    ("8 weeks", 2),
    ("no duration here", None),
])
def test_stated_durations_are_parsed(text, expected):
    """Viktor Sharat's resume states tenure as durations with no dates at all."""
    assert parse_duration_to_months(text) == expected


# ------------------------------------------------------------ document cleaning
def test_merged_table_cells_are_deduplicated():
    """Two resumes repeat every table row up to four times via merged cells."""
    assert _dedupe_row_cells(["A", "A", "A", "B", ""]) == ["A", "B"]


def test_ligature_damage_is_repaired():
    """Omar El-Hassan's PDF loses the 'ti' ligature in every affected word."""
    cleaned, repairs = clean_text("Quan\ufffdta\ufffdve Developer")
    assert cleaned == "Quantitative Developer"
    assert any("U+FFFD" in r for r in repairs)


def test_smart_punctuation_is_normalised():
    cleaned, _ = clean_text("Analyst \u2013 TMT \u2018coverage\u2019")
    assert cleaned == "Analyst - TMT 'coverage'"


# ------------------------------------------------------------ validation rules
def _profile(**overrides) -> CandidateProfile:
    base = dict(
        source_file="test.docx", full_name="Test Candidate",
        email="test@example.com", phone="+1 555 0100",
        region="North America", primary_strategy_type="Fundamental",
        sectors_covered=["Healthcare"], roles=[], education=[],
    )
    base.update(overrides)
    return CandidateProfile(**base)


def test_self_reported_experience_conflict_is_flagged():
    """Priya Nakamura claims 9 years; her dates imply 12.7."""
    profile = _profile(
        self_reported_years_experience=9,
        roles=[Role(employer="Firm A", start_date="2013-10", end_date="present", is_current=True)],
    )
    result = enrich(profile)
    assert result.computed_years_experience > 12
    assert any("disagrees" in f for f in result.data_quality_flags)
    assert "self-report disputed" in result.years_experience_source


def test_overlapping_roles_at_one_employer_are_flagged():
    """Omar El-Hassan's internship overlaps his full-time role at BNP Paribas."""
    profile = _profile(roles=[
        Role(employer="BNP Paribas", start_date="2022-05", end_date="present", is_current=True),
        Role(employer="BNP Paribas", start_date="2022-04", end_date="2022-08", is_internship=True),
    ])
    flags = enrich(profile).data_quality_flags
    assert any("Overlapping roles" in f and "same employer" in f for f in flags)


def test_impossible_dates_are_flagged():
    profile = _profile(roles=[Role(employer="Firm A", start_date="2020-06", end_date="2019-01")])
    assert any("Impossible dates" in f for f in enrich(profile).data_quality_flags)


def test_malformed_email_is_flagged():
    """Marcus Chen-Rodriguez's email has no top-level domain."""
    assert any("Malformed email" in f for f in enrich(_profile(email="rchen@hotmail")).data_quality_flags)


def test_undergraduate_degree_at_graduate_only_school_is_flagged():
    """Ryan Patel and Vikram Shah both attribute a bachelor's to a graduate-only school."""
    profile = _profile(education=[
        Education(institution="Northwestern University - Kellogg School of Management",
                  degree="Bachelor of Science in Business Administration", end_year=2014),
    ])
    assert any("graduate-only school" in f for f in enrich(profile).data_quality_flags)


def test_employer_contradiction_inside_a_role_is_flagged():
    """Marina Silva Costa's Bain role credits McKinsey's case competition."""
    profile = _profile(roles=[Role(
        employer="Bain & Company", start_date="2016", end_date="2019",
        highlights=["Led launch of McKinsey's first case competition in Brazil"],
    )])
    assert any("possible employer mismatch" in f for f in enrich(profile).data_quality_flags)


def test_study_period_gaps_are_annotated_not_treated_as_unexplained():
    """An MBA is an explanation for a gap, not a red flag."""
    profile = _profile(
        roles=[
            Role(employer="Consultancy", start_date="2016", end_date="2019"),
            Role(employer="Asset Manager", start_date="2021-08", end_date="present", is_current=True),
        ],
        education=[Education(institution="MIT Sloan", degree="MBA", start_year=2019, end_year=2021)],
    )
    gaps = enrich(profile).career_gaps
    assert len(gaps) == 1
    assert "overlaps" in gaps[0] and "MBA" in gaps[0]


def test_internship_summers_do_not_create_unemployment_gaps():
    profile = _profile(roles=[
        Role(employer="Fund", start_date="2016-05", end_date="2016-08", is_internship=True),
        Role(employer="Employer", start_date="2017-07", end_date="present", is_current=True),
    ])
    assert enrich(profile).career_gaps == []


def test_missing_current_role_is_reported():
    """Chen Li has no role recorded after September 2023."""
    result = enrich(_profile(roles=[Role(employer="Bank", start_date="2021-06", end_date="2023-09")]))
    assert result.is_currently_employed is False
    assert any("Not currently employed" in g for g in result.career_gaps)


def test_duplicate_flags_are_not_double_counted():
    """A duplicated education entry must not penalise the quality score twice."""
    edu = Education(institution="Kellogg School of Management", degree="Bachelor of Science", end_year=2014)
    single = enrich(_profile(education=[edu]))
    doubled = enrich(_profile(education=[edu, edu]))
    graduate_flags = [f for f in doubled.data_quality_flags if "graduate-only" in f]
    assert len(graduate_flags) == 1
    assert doubled.data_quality_score <= single.data_quality_score


def test_degree_tier_picks_the_highest_qualification():
    profile = _profile(education=[
        Education(institution="School", degree="B.M.S", end_year=2007),
        Education(institution="Medical College", degree="M.B.B.S", end_year=2010),
        Education(institution="Business School", degree="PGDM (Finance)", end_year=2012),
    ])
    assert enrich(profile).highest_degree_tier == "Doctorate / Medical"


def test_internships_are_excluded_from_the_experience_total():
    profile = _profile(roles=[
        Role(employer="Fund", start_date="2019-01", end_date="2019-12", is_internship=True),
        Role(employer="Employer", start_date="2020-01", end_date="2021-12"),
    ])
    result = enrich(profile)
    assert result.computed_years_experience > result.computed_years_excluding_internships


def test_clean_record_scores_near_one():
    profile = _profile(
        roles=[Role(employer="Fund", start_date="2018-01", end_date="present", is_current=True,
                    sectors=["Healthcare"])],
        education=[Education(institution="University", degree="MBA", start_year=2015, end_year=2017)],
    )
    result = enrich(profile)
    assert result.data_quality_score >= 0.9
    assert result.data_quality_flags == []

Overwriting tests/test_pipeline.py


In [13]:
!python -m pytest tests/ -q

.........................                                                [100%]


25 passed in 0.28s


### Running the pipeline

`--seed-cache` writes the committed extraction output (`data/reference_extractions.json`)
into the cache so this notebook is reproducible without credentials. Set
`refresh=True` with an API key exported to bypass the cache entirely and call the live
API for all 10 resumes.

In [14]:
import pipeline

candidates = pipeline.run(refresh=False, seed=True)
print(f"\n{len(candidates)} candidates parsed and validated.")

Seeding offline cache from reference extractions...


  [seed ] Chen-Li-Alex.docx
  [seed ] MARINA-SILVA-COSTA.docx
  [seed ] Marcus-Chen-Rodriguez-Resume.docx
  [seed ] Michael-Rodriguez-CFA.docx
  [seed ] Omar-El-Hassan-202405.pdf
  [seed ] Priya-Nakamura_sellside_healthcare_RLTM.docx
  [seed ] RYAN-PATEL-Resume.pdf
  [seed ] Vikram-Shah.docx
  [seed ] Viktor-Sharat.docx
  [seed ] Zara-Al-Rashid.docx

1. Extracting text from resumes in /home/user/workspace/millennium_case/resumes


  Chen-Li-Alex.docx                                docx   4,270 chars
  MARINA-SILVA-COSTA.docx                          docx   3,499 chars
  Marcus-Chen-Rodriguez-Resume.docx                docx   3,407 chars
  Michael-Rodriguez-CFA.docx                       docx   3,608 chars
  Omar-El-Hassan-202405.pdf                        pdf    1,641 chars | repairs: extracted 1 PDF page(s); expanded ligature 'ﬀ' -> 'ff'; expanded ligature 'ﬃ' -> 'ffi'; repaired 29 token(s) containing U+FFFD replacement characters
  Priya-Nakamura_sellside_healthcare_RLTM.docx     docx   5,188 chars
  RYAN-PATEL-Resume.pdf                            pdf    4,954 chars | repairs: extracted 2 PDF page(s)
  Vikram-Shah.docx                                 docx   3,880 chars
  Viktor-Sharat.docx                               docx   3,057 chars | repairs: removed 12 duplicate table cell value(s) from merged cells
  Zara-Al-Rashid.docx                              docx   3,401 chars | repairs: removed 4 duplicate tab

---
## 6. Parsed output

Four exports, each serving a different consumer: nested JSON for the application, a
flat candidate table for Excel and BI, a role-level table for tenure and firm analysis,
and a flag-level table for review triage.

In [15]:
flat = pd.read_csv(Path(config.OUTPUT_DIR) / "candidates.csv")
flat[["full_name", "region", "current_employer", "strategy_type", "market_side",
      "seniority_level", "years_experience", "sectors_covered", "highest_degree_tier",
      "has_cfa", "data_quality_score"]]

,full_name,region,current_employer,strategy_type,market_side,seniority_level,years_experience,sectors_covered,highest_degree_tier,has_cfa,data_quality_score
0,Chen Li (Alex),Asia-Pacific,NaN,Systematic / Quantitative,Buy-Side,Analyst,4.4,Technology; Healthcare; Financial Services,Master's / MBA,False,0.80
1,Marina Silva Costa,Europe,Vanguard Group,Fundamental,Buy-Side,Analyst,9.1,Healthcare; Consumer; Media & Telecom; Industrials; Financial Services,Master's / MBA,False,0.80
2,Marcus Chen-Rodriguez,North America,Coatue Management,Fundamental,Buy-Side,Senior Analyst,10.9,Healthcare; Credit,Bachelor's,False,0.80
3,Michael Rodriguez,North America,Fidelity Asset Management,Fundamental,Buy-Side,Analyst,8.9,Technology; Media & Telecom; Consumer; Credit; Industrials; Financial Services,Bachelor's,True,0.91
4,Omar El-Hassan,Europe,BNP Paribas CIB,Systematic / Quantitative,Sell-Side,Analyst,4.5,Macro / Rates & FX; Credit,Master's / MBA,False,0.77
5,Priya Nakamura,Asia-Pacific,ICICI Securities,Fundamental,Sell-Side,Lead Analyst,12.7,Healthcare; Materials,Master's / MBA,False,0.36
6,Ryan Patel,North America,Meridian Capital Partners,Fundamental,Buy-Side,Portfolio Manager,8.3,Multi-Sector / Generalist; Consumer; Technology; Media & Telecom; Healthcare; Industrials,Bachelor's,False,0.55
7,Vikram Shah,North America,Cinctive Capital Management,Fundamental,Buy-Side,Analyst,12.0,Technology; Media & Telecom; Consumer; Financial Services,Bachelor's,True,0.72
8,Viktor Sharat,Asia-Pacific,NaN,Fundamental,Sell-Side,Lead Analyst,11.5,Healthcare; Energy; Industrials; Consumer; Materials; Multi-Sector / Generalist,Master's / MBA,False,0.55
9,Dr. Zara Al-Rashid,Asia-Pacific,Meridian Research Partners,Fundamental,Sell-Side,Lead Analyst,10.7,Healthcare,Doctorate / Medical,False,0.47


In [16]:
# One record in full, to show extraction depth and the provenance fields.
print(json.dumps(candidates[6].model_dump(), indent=2, default=str)[:4000])

{
  "source_file": "RYAN-PATEL-Resume.pdf",
  "source_agency": null,
  "full_name": "Ryan Patel",
  "honorific": null,
  "email": "ryan.patel0403@gmail.com",
  "phone": "+1 (516) 523-3113",
  "location_city": "Brooklyn, New York",
  "location_country": "United States",
  "region": "North America",
  "current_employer": "Meridian Capital Partners",
  "current_title": "Investment Professional, Generalist - Soft Catalyst & Fundamental Long/Short",
  "primary_strategy_type": "Fundamental",
  "primary_market_side": "Buy-Side",
  "primary_firm_type": "Hedge Fund",
  "seniority_level": "Portfolio Manager",
  "sectors_covered": [
    "Multi-Sector / Generalist",
    "Consumer",
    "Technology",
    "Media & Telecom",
    "Healthcare",
    "Industrials"
  ],
  "sector_specialisation_detail": [
    "Generalist soft catalyst and fundamental long/short mandate",
    "Consumer and Technology sub-portfolio within a $4.2bn gross book",
    "Private equity across Business & Technology Services, Consu

In [17]:
roles = pd.read_csv(Path(config.OUTPUT_DIR) / "candidates_roles.csv")
print(f"{len(roles)} roles extracted across {roles['candidate_id'].nunique()} candidates")
roles[["full_name", "employer", "title", "start_date", "end_date", "duration_stated",
       "firm_type", "market_side", "sectors", "coverage_universe_size"]].head(20)

48 roles extracted across 10 candidates


,full_name,employer,title,start_date,end_date,duration_stated,firm_type,market_side,sectors,coverage_universe_size
0,Chen Li (Alex),Bank of China,Investment Analyst,2021-06,2023-09,NaN,Commercial / Corporate Bank,Buy-Side,Technology; Financial Services,NaN
1,Chen Li (Alex),Bank of China,Investment Intern,2019-12,2021-05,NaN,Commercial / Corporate Bank,Buy-Side,Technology; Healthcare,35.0
2,Chen Li (Alex),Chinese University of Hong Kong,Research Assistant (Economics Area),2019-03,2021-03,NaN,Academic / Research,Academic,Multi-Sector / Generalist,NaN
3,Chen Li (Alex),Meridian Asia Capital,Finance Market Department Intern (Chinese Bond and Equity Market),2019-07,2019-08,NaN,Other,Unknown,Financial Services; Credit,NaN
4,Marina Silva Costa,Vanguard Group,Equity Research Analyst,2021-08,present,NaN,Asset Manager,Buy-Side,Healthcare; Consumer; Media & Telecom,NaN
5,Marina Silva Costa,Vanguard Group,Equity Research Summer Analyst,2020-06,2020-08,NaN,Asset Manager,Buy-Side,Industrials; Utilities,NaN
6,Marina Silva Costa,Bain & Company,Business Analyst,2016,2019,NaN,Consulting,Corporate,Consumer; Industrials; Financial Services,NaN
7,Marcus Chen-Rodriguez,Coatue Management,"Healthcare Analyst, Investment Team",2019-10,present,NaN,Hedge Fund,Buy-Side,Healthcare,NaN
8,Marcus Chen-Rodriguez,Goldman Sachs,"Senior Associate, Investment Team",2019-01,2019-09,NaN,Investment Bank - Banking / Markets,Private Markets,Healthcare; Credit,NaN
9,Marcus Chen-Rodriguez,Goldman Sachs,"Associate, Investment Team",2017-04,2018-12,NaN,Investment Bank - Banking / Markets,Private Markets,Healthcare; Credit,NaN


### What validation caught

This is the part of the exercise I would most want to discuss. The sample resumes
contain deliberate inconsistencies, and a platform that ingests them without comment
would present a false picture of the talent pool.

In [18]:
quality = pd.read_csv(Path(config.OUTPUT_DIR) / "data_quality_report.csv")
print(f"{len(quality)} findings across {quality['candidate_id'].nunique()} candidates")
print(f"  raised by validation rules : {(quality['flag_origin'] == 'rule').sum()}")
print(f"  reported by the model      : {(quality['flag_origin'] == 'model').sum()}")

rules = quality[quality["flag_origin"] == "rule"]
for name, group in rules.groupby("full_name"):
    print(f"\n{name}")
    for f in group["flag"]:
        print(f"  - {f}")

79 findings across 10 candidates
  raised by validation rules : 26
  reported by the model      : 53

Chen Li (Alex)
  - Overlapping roles (different employers, ~15 months): Bank of China [2019-12-2021-05] and Chinese University of Hong Kong [2019-03-2021-03]
  - Career gap: Not currently employed - 35 months since 2023-09

Marcus Chen-Rodriguez
  - Malformed email address: rchen@hotmail
  - Career gap: 25 month gap between 2015-03 and 2017-04 - unexplained

Marina Silva Costa
  - Role at Bain & Company contains a bullet referring to employment at 'mckinsey' - possible employer mismatch
  - Career gap: 20 month gap between 2019-12 and 2021-08 - overlaps Master of Business Administration (MBA) at MIT Sloan School of Management

Omar El-Hassan
  - Overlapping roles (same employer, ~3 months): BNP Paribas CIB [2022-05-present] and BNP Paribas CIB [2022-04-2022-08]

Priya Nakamura
  - Self-reported experience (9 yrs) disagrees with experience computed from dates (12.7 yrs) by 3.7 yrs
  - N

The findings that matter most for a hiring decision:

| Candidate | Finding | Why it matters |
| --- | --- | --- |
| Priya Nakamura | Profile claims 9 years of healthcare coverage; the dated roles imply 12.7 | The platform reports the computed figure and flags the disagreement rather than choosing one |
| Priya Nakamura | A Jardine Lloyd Thompson role whose bullet says "started my journey as a Lead Analyst at Anand Rathi" | Employer history cannot be taken at face value |
| Omar El-Hassan | A full-time role from May 2022 overlapping an internship at the same firm from April to August 2022 | Chronologically impossible as written |
| Ryan Patel | A Millennium role titled "North53 Capital" whose bullet says the book was "for Vertex Capital" | Two different entities inside one entry |
| Ryan Patel | A Bachelor of Science attributed to Columbia Business School | CBS is graduate-only, so the degree attribution is wrong |
| Vikram Shah | Kellogg listed twice, and a bachelor's degree attributed to Kellogg | Kellogg is graduate-only; also a formatting duplication |
| Marina Silva Costa | A Bain & Company role crediting the launch of "McKinsey's first case competition" | Internal contradiction in the same entry |
| Marcus Chen-Rodriguez | Email `rchen@hotmail` with no top-level domain | Unusable contact detail, and the local part does not match the stated first name |
| Chen Li (Alex) | No role after September 2023 | Currently unplaced, which changes how BD would approach them |
| Viktor Sharat | Tenure given only as durations, with no dates anywhere | Chronology, current status and gaps are genuinely unknowable and are reported as such |
| Priya Nakamura | A "Red Lane Talent Management" watermark | Agency-sourced document, captured as provenance metadata rather than discarded as noise |

Three candidates have no contact details at all, which is a practical sourcing
obstacle worth surfacing before someone tries to reach out.

In [19]:
# Corpus-level view: the shape of the talent pool the BD team actually has.
import validation
print(json.dumps(validation.corpus_quality_summary(candidates), indent=2))

print("\nRegion x strategy:")
print(pd.crosstab(flat["region"], flat["strategy_type"]))
print("\nSector coverage (candidates per sector):")
print(flat["sectors_covered"].str.split("; ").explode().value_counts())
print("\nExperience:")
print(flat["years_experience"].describe().round(1))

{
  "candidates": 10,
  "with_flags": 10,
  "total_flags": 74,
  "mean_quality_score": 0.673,
  "high_confidence": 1,
  "currently_employed": 8
}

Region x strategy:


strategy_type  Fundamental  Systematic / Quantitative
region                                               
Asia-Pacific             3                          1
Europe                   1                          1
North America            4                          0

Sector coverage (candidates per sector):
sectors_covered
Healthcare                   7
Consumer                     5
Technology                   4
Financial Services           4
Media & Telecom              4
Industrials                  4
Credit                       3
Materials                    2
Multi-Sector / Generalist    2
Macro / Rates & FX           1
Energy                       1
Name: count, dtype: int64

Experience:
count    10.0
mean      9.3
std       2.9
min       4.4
25%       8.5
50%       9.9
75%      11.4
max      12.7
Name: years_experience, dtype: float64


---
## 7. The Streamlit application

**Link:** the deployed application is live at **https://millennium-ds-case-study-hqvfr4iirqkxre4kgk5vrj.streamlit.app/#candidate-resume-search-platform**, running the exact code in this notebook from the project repository.

Run it locally with `streamlit run app.py` (defaults to http://localhost:8501). Nothing
in the code differs between local and hosted execution; deploying to an internal host is
a configuration change, not a code change - see section 9.

### Interface design

BD does not browse candidates, it fills mandates. So the app is organised around a
**requisition** - region, investment approach, sector coverage and an experience band -
with five preset mandates modelled on the search dimensions named in the brief.
Requisitions are deep-linkable (`?req=2`) so a recruiter can send a colleague the exact
view rather than a description of which filters to set.

**Search and filtering.** Fourteen filters: region, investment approach, sector
coverage with any/all logic, experience band, seniority, firm type, market side,
highest degree, language, CFA, medical degree, currently-employed, minimum data-quality
score, and comma-separated keyword search across the full resume text.

**Transparent scoring.** The match score is a visible weighted sum over five
components - sector fit, region, strategy, experience and credentials - each a
documented ratio, with the weights exposed as sliders and a per-candidate contribution
chart. A hiring manager can always be told exactly why one candidate ranks above
another. This matters beyond aesthetics: an opaque ranking in a hiring context is a
liability, and a score nobody can explain will not be trusted or used.

Two deliberate softening choices. Experience bands drive the score with linear decay
outside the band rather than acting as a hard cut, because excluding someone for being
six months outside a range loses good people; a checkbox makes it a hard filter when a
recruiter really wants one. And when no requisition criteria are set, the app ranks by
experience and shows data quality instead of a score, because with every dimension set
to "Any" every candidate scores near-perfectly and the ranking would be meaningless.

**Four views.** Search results (cards or table, with CSV shortlist export), side-by-side
comparison with a normalised radar chart, talent-pool analytics, and a data-quality
review tab.

**Performance.** Data loads once through `st.cache_data`, because Streamlit re-runs the
whole script on every widget change. Filtering is a vectorised boolean mask over a
pandas frame rather than per-row Python, so the interaction model holds as the corpus
grows.

### Search results and match scoring

![Search results](outputs/shot_01_results.png)

Scoring against the "US Healthcare Fundamental Analyst" mandate. Ryan Patel scores 92
and Marcus Chen-Rodriguez 91; both are visibly tagged with their data-quality flag
counts, so a recruiter knows to check before acting.

### Side-by-side comparison

![Comparison](outputs/shot_02_compare.png)

### Talent pool analytics

![Insights](outputs/shot_03_insights.png)

Sector coverage by region, fundamental versus systematic mix, experience distribution,
current firm type, coverage universe against experience, and tooling frequency - plus a
coverage-gap table showing where the pipeline is thin for the sectors the current
requisition needs.

### Data quality review

![Data quality](outputs/shot_04_quality.png)

Every finding, labelled by whether a validation rule or the extraction model raised it,
filterable and exportable.

In [20]:
%%writefile app.py
"""Millennium BD - Candidate Resume Search Platform (Streamlit).

Run:
    streamlit run app.py

Design principles
-----------------
* **Requisition-first.** BD does not browse candidates, it fills mandates. The app is
  therefore organised around a requisition (region + strategy + sectors + experience
  band) with preset mandates, and every candidate is scored against it.
* **Transparent scoring.** The match score is a visible weighted sum with a per-
  component breakdown and adjustable weights. A black-box ranking cannot be defended
  to a hiring manager, and an opaque score in a hiring context is a liability.
* **Decision support, not automation.** Nothing is filtered out silently. Data-quality
  flags travel with every candidate, and the source resume text is one click away, so
  a human always verifies before acting.
* **Performance.** Data loads once via `st.cache_data`; filtering runs on a vectorised
  pandas frame rather than per-row Python, so the interaction model holds as the corpus
  grows from 10 to 10,000+ records.
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

ROOT = Path(__file__).parent
sys.path.insert(0, str(ROOT / "src"))

import config  # noqa: E402

st.set_page_config(
    page_title="Millennium BD - Candidate Search",
    page_icon="\U0001F50D",
    layout="wide",
    initial_sidebar_state="expanded",
)

PRIMARY = "#0B3C5D"
ACCENT = "#328CC1"
PALETTE = ["#0B3C5D", "#328CC1", "#1D7874", "#D9B310", "#8C6A5D", "#5B7553", "#A23B72", "#6C757D"]

st.markdown(
    f"""
    <style>
      .block-container {{padding-top: 2rem; padding-bottom: 3rem; max-width: 1500px;}}
      h1, h2, h3 {{color: {PRIMARY}; letter-spacing: -0.01em;}}
      div[data-testid="stMetricValue"] {{font-size: 1.55rem; color: {PRIMARY};}}
      .cand-card {{
        border: 1px solid #E3E7EB; border-left: 4px solid {ACCENT}; border-radius: 8px;
        padding: 1rem 1.15rem; margin-bottom: 0.85rem; background: #FFFFFF;
      }}
      .cand-name {{font-size: 1.06rem; font-weight: 650; color: {PRIMARY};}}
      .cand-sub {{color: #55606B; font-size: 0.88rem; margin-top: 0.1rem;}}
      .pill {{
        display: inline-block; padding: 0.13rem 0.6rem; border-radius: 999px;
        font-size: 0.74rem; margin: 0.12rem 0.22rem 0.12rem 0; background: #EEF3F7; color: {PRIMARY};
      }}
      .pill-warn {{background: #FDF2E3; color: #8A5A00;}}
      .pill-good {{background: #E8F4EC; color: #14622F;}}
      .score {{font-size: 1.45rem; font-weight: 700; color: {ACCENT}; text-align: right;}}
      .muted {{color: #6B7680; font-size: 0.82rem;}}
    </style>
    """,
    unsafe_allow_html=True,
)


# ------------------------------------------------------------------ data layer
@st.cache_data(show_spinner=False)
def load_candidates() -> tuple[pd.DataFrame, list[dict]]:
    """Load parsed candidates once per session.

    Cached because every filter change re-runs the whole script top to bottom in
    Streamlit. Without caching, a corpus of any real size would re-read and re-parse
    JSON on each keystroke.
    """
    path = Path(config.CANDIDATES_JSON)
    if not path.exists():
        return pd.DataFrame(), []
    records = json.loads(path.read_text())

    rows = []
    for c in records:
        rows.append({
            "candidate_id": c["candidate_id"],
            "name": (f"{c['honorific']} " if c.get("honorific") else "") + c["full_name"],
            "region": c["region"],
            "location": ", ".join(x for x in [c.get("location_city"), c.get("location_country")] if x) or "Not stated",
            "current_employer": c.get("current_employer") or "No current employer stated",
            "current_title": c.get("current_title") or "No current title stated",
            "employed": bool(c.get("is_currently_employed")),
            "strategy": c["primary_strategy_type"],
            "market_side": c["primary_market_side"],
            "firm_type": c["primary_firm_type"],
            "seniority": c["seniority_level"],
            "sectors": c.get("sectors_covered", []),
            "markets": c.get("geographic_markets_covered", []),
            "years": float(c.get("computed_years_experience") or 0),
            "years_ex_int": float(c.get("computed_years_excluding_internships") or 0),
            "self_reported_years": c.get("self_reported_years_experience"),
            "coverage": c.get("max_coverage_universe"),
            "degree_tier": c.get("highest_degree_tier", "Unknown"),
            "highest_degree": c.get("highest_degree"),
            "has_cfa": bool(c.get("has_cfa")),
            "has_md": bool(c.get("has_medical_degree")),
            "languages": c.get("languages_spoken", []),
            "programming": c.get("programming_languages", []),
            "tools": c.get("tools_and_platforms", []),
            "employers": c.get("employer_list", []),
            "n_employers": c.get("n_employers", 0),
            "quality": float(c.get("data_quality_score") or 0),
            "n_flags": len(c.get("data_quality_flags", [])),
            "confidence": c.get("extraction_confidence", "medium"),
            "agency": c.get("source_agency") or "",
            "source_file": c["source_file"],
            "searchable": c.get("searchable_text", ""),
        })
    return pd.DataFrame(rows), records


@st.cache_data(show_spinner=False)
def load_raw_text(source_file: str) -> str:
    stem = Path(source_file).stem
    path = Path(config.RAW_TEXT_DIR) / f"{stem}.txt"
    return path.read_text(encoding="utf-8") if path.exists() else "Source text not available."


def record_by_id(records: list[dict], cid: str) -> dict:
    return next(c for c in records if c["candidate_id"] == cid)


# --------------------------------------------------------------- match scoring
PRESETS: dict[str, dict] = {
    "-- No requisition (browse all) --": {},
    "US Healthcare Fundamental Analyst (5-10 yrs)": {
        "region": "North America", "strategy": "Fundamental", "sectors": ["Healthcare"],
        "min_years": 5.0, "max_years": 10.0, "seniority": ["Analyst", "Senior Analyst", "Associate"],
    },
    "US TMT Fundamental L/S Analyst (4-12 yrs)": {
        "region": "North America", "strategy": "Fundamental",
        "sectors": ["Technology", "Media & Telecom"], "min_years": 4.0, "max_years": 12.0,
        "seniority": ["Analyst", "Senior Analyst", "Associate"],
    },
    "Europe Systematic / Quant Researcher (2-8 yrs)": {
        "region": "Europe", "strategy": "Systematic / Quantitative",
        "sectors": ["Macro / Rates & FX", "Credit"], "min_years": 2.0, "max_years": 8.0,
    },
    "APAC Healthcare Research Analyst (6-15 yrs)": {
        "region": "Asia-Pacific", "strategy": "Fundamental", "sectors": ["Healthcare"],
        "min_years": 6.0, "max_years": 15.0,
    },
    "Global Credit / Macro Analyst (3-10 yrs)": {
        "strategy": "Any", "sectors": ["Credit", "Macro / Rates & FX"],
        "min_years": 3.0, "max_years": 10.0,
    },
}

DEFAULT_WEIGHTS = {"sector": 35, "region": 20, "strategy": 20, "experience": 15, "credentials": 10}


def score_candidate(row: pd.Series, req: dict, weights: dict[str, int]) -> tuple[float, dict[str, float]]:
    """Transparent weighted match score in [0, 100] plus its component breakdown.

    Every component is a documented ratio, so a recruiter can always be told exactly
    why one candidate outranks another. Nothing here is learned or hidden.
    """
    parts: dict[str, float] = {}

    # Sector: proportion of requested sectors the candidate actually covers. A
    # generalist mandate counts as full coverage.
    wanted = set(req.get("sectors") or [])
    if wanted:
        have = set(row["sectors"])
        overlap = len(wanted & have)
        if "Multi-Sector / Generalist" in have and overlap == 0:
            parts["sector"] = 0.55           # credible but unproven in the target sector
        else:
            parts["sector"] = overlap / len(wanted)
    else:
        parts["sector"] = 1.0

    # Region: exact match, with partial credit for a candidate who researches the
    # target market from elsewhere.
    target_region = req.get("region")
    if target_region and target_region != "Any":
        if row["region"] == target_region:
            parts["region"] = 1.0
        else:
            market_blob = " ".join(row["markets"]).lower()
            hint = {"North America": ["united states", "us", "north america"],
                    "Europe": ["europe", "united kingdom", "emea", "france", "germany"],
                    "Asia-Pacific": ["asia", "china", "india", "japan", "hong kong"],
                    "Latin America": ["latam", "latin america", "brazil"],
                    "Middle East & Africa": ["middle east", "africa", "emea"]}.get(target_region, [])
            parts["region"] = 0.5 if any(h in market_blob for h in hint) else 0.0
    else:
        parts["region"] = 1.0

    # Strategy: hybrid profiles get partial credit for either mandate.
    target_strategy = req.get("strategy")
    if target_strategy and target_strategy != "Any":
        if row["strategy"] == target_strategy:
            parts["strategy"] = 1.0
        elif row["strategy"] == "Hybrid":
            parts["strategy"] = 0.7
        else:
            parts["strategy"] = 0.0
    else:
        parts["strategy"] = 1.0

    # Experience: full credit inside the band, decaying linearly outside it, because a
    # candidate one year outside a band is not a non-match.
    lo, hi = req.get("min_years"), req.get("max_years")
    yrs = row["years"]
    if lo is None and hi is None:
        parts["experience"] = 1.0
    else:
        lo = lo if lo is not None else 0.0
        hi = hi if hi is not None else 60.0
        if lo <= yrs <= hi:
            parts["experience"] = 1.0
        else:
            distance = (lo - yrs) if yrs < lo else (yrs - hi)
            parts["experience"] = max(0.0, 1.0 - distance / 5.0)

    # Credentials: a small bonus pool, never a gate. Seniority alignment is included
    # here because titles are noisy across firms and geographies.
    cred = 0.0
    cred += 0.35 if row["degree_tier"] in {"Master's / MBA", "Doctorate / Medical"} else 0.15
    cred += 0.25 if row["has_cfa"] else 0.0
    cred += 0.15 if (row["has_md"] and "Healthcare" in (req.get("sectors") or [])) else 0.0
    wanted_seniority = req.get("seniority") or []
    cred += 0.25 if (not wanted_seniority or row["seniority"] in wanted_seniority) else 0.0
    parts["credentials"] = min(cred, 1.0)

    total_w = sum(weights.values()) or 1
    score = sum(parts[k] * weights.get(k, 0) for k in parts) / total_w * 100
    contributions = {k: round(parts[k] * weights.get(k, 0) / total_w * 100, 1) for k in parts}
    return round(score, 1), contributions


# ------------------------------------------------------------------- rendering
def pill(text: str, kind: str = "") -> str:
    cls = {"warn": "pill pill-warn", "good": "pill pill-good"}.get(kind, "pill")
    return f'<span class="{cls}">{text}</span>'


def candidate_card(row: pd.Series, contributions: dict[str, float] | None, show_score: bool) -> None:
    pills = [pill(s) for s in row["sectors"][:5]]
    pills.append(pill(row["strategy"], "good" if row["strategy"] != "Unclear" else "warn"))
    pills.append(pill(row["market_side"]))
    if row["has_cfa"]:
        pills.append(pill("CFA", "good"))
    if row["has_md"]:
        pills.append(pill("MD / MBBS", "good"))
    if not row["employed"]:
        pills.append(pill("Not currently employed", "warn"))
    if row["n_flags"]:
        pills.append(pill(f"{row['n_flags']} data flags", "warn"))
    if isinstance(row["agency"], str) and row["agency"].strip():
        pills.append(pill(f"via {row['agency']}", "warn"))

    left, right = st.columns([5, 1])
    with left:
        st.markdown(
            f"""<div class="cand-card">
              <div class="cand-name">{row['name']}</div>
              <div class="cand-sub">{row['current_title']} &middot; {row['current_employer']}</div>
              <div class="cand-sub">{row['location']} &middot; {row['region']} &middot;
                 {row['years']:.1f} yrs experience &middot; {row['seniority']} &middot;
                 {row['degree_tier']}</div>
              <div style="margin-top:0.5rem">{''.join(pills)}</div>
            </div>""",
            unsafe_allow_html=True,
        )
    with right:
        if show_score and contributions is not None:
            st.markdown(f'<div class="score">{sum(contributions.values()):.0f}</div>'
                        f'<div class="muted" style="text-align:right">match score</div>',
                        unsafe_allow_html=True)
        else:
            st.markdown(f'<div class="score">{row["quality"]:.2f}</div>'
                        f'<div class="muted" style="text-align:right">data quality</div>',
                        unsafe_allow_html=True)


def candidate_detail(rec: dict, contributions: dict[str, float] | None = None) -> None:
    name = (f"{rec['honorific']} " if rec.get("honorific") else "") + rec["full_name"]
    st.markdown(f"### {name}")

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Experience (computed)", f"{rec['computed_years_experience']:.1f} yrs")
    c2.metric("Employers", rec.get("n_employers", 0))
    c3.metric("Max coverage universe", rec.get("max_coverage_universe") or "n/a")
    c4.metric("Data quality", f"{rec['data_quality_score']:.2f}")

    if contributions:
        st.markdown("**Why this match score**")
        fig = go.Figure(go.Bar(
            x=list(contributions.values()), y=[k.title() for k in contributions],
            orientation="h", marker_color=ACCENT,
            text=[f"{v:.1f}" for v in contributions.values()], textposition="outside",
        ))
        fig.update_layout(height=210, margin=dict(l=0, r=30, t=6, b=0),
                          xaxis_title="points contributed", plot_bgcolor="white")
        st.plotly_chart(fig, use_container_width=True)

    tabs = st.tabs(["Career", "Education & skills", "Data quality", "Source resume"])

    with tabs[0]:
        rows = []
        for r in rec["roles"]:
            period = (f"{r['start_date'] or '?'} to {r['end_date'] or ('present' if r['is_current'] else '?')}"
                      if r.get("start_date") else (r.get("duration_stated") or "dates not stated"))
            rows.append({
                "Employer": r["employer"], "Title": r.get("title"), "Period": period,
                "Firm type": r["firm_type"], "Side": r["market_side"], "Style": r["strategy_type"],
                "Sectors": ", ".join(r.get("sectors", [])),
                "Coverage": r.get("coverage_universe_size"),
                "Intern": "yes" if r.get("is_internship") else "",
            })
        st.dataframe(pd.DataFrame(rows), use_container_width=True, hide_index=True)
        for r in rec["roles"]:
            if r.get("highlights"):
                with st.expander(f"{r['employer']} - {r.get('title') or 'role'} highlights"):
                    for h in r["highlights"]:
                        st.markdown(f"- {h}")

    with tabs[1]:
        if rec.get("education"):
            st.dataframe(pd.DataFrame([{
                "Institution": e["institution"], "Degree": e.get("degree"),
                "Field": e.get("field_of_study"), "Completed": e.get("end_year"),
                "Grade": e.get("gpa"), "Honors": e.get("honors"),
            } for e in rec["education"]]), use_container_width=True, hide_index=True)
        cols = st.columns(2)
        with cols[0]:
            st.markdown("**Certifications**")
            st.write(", ".join(rec.get("certifications") or []) or "None stated")
            st.markdown("**Programming**")
            st.write(", ".join(rec.get("programming_languages") or []) or "None stated")
            st.markdown("**Languages**")
            st.write(", ".join(rec.get("languages_spoken") or []) or "None stated")
        with cols[1]:
            st.markdown("**Tools and platforms**")
            st.write(", ".join(rec.get("tools_and_platforms") or []) or "None stated")
            st.markdown("**Sector detail (verbatim)**")
            for d in rec.get("sector_specialisation_detail") or []:
                st.markdown(f"- {d}")
            if rec.get("publications"):
                st.markdown("**Publications**")
                for p in rec["publications"]:
                    st.markdown(f"- {p}")

    with tabs[2]:
        st.caption(f"Model extraction confidence: **{rec['extraction_confidence']}**"
                   + (f" &middot; agency-sourced document: **{rec['source_agency']}**" if rec.get("source_agency") else ""))
        rules = [f for f in rec["data_quality_flags"] if not f.startswith("[model]")]
        model_notes = [f.replace("[model] ", "") for f in rec["data_quality_flags"] if f.startswith("[model]")]
        if rec.get("career_gaps"):
            st.markdown("**Career continuity**")
            for g in rec["career_gaps"]:
                st.warning(g, icon="\u26A0\uFE0F")
        if rules:
            st.markdown("**Automated validation findings**")
            for f in rules:
                st.markdown(f"- {f}")
        if model_notes:
            st.markdown("**Ambiguities reported by the extraction model**")
            for f in model_notes:
                st.markdown(f"- {f}")
        if not (rules or model_notes or rec.get("career_gaps")):
            st.success("No data-quality issues detected.")

    with tabs[3]:
        st.caption(f"Extracted text from `{rec['source_file']}` - the exact input the model received.")
        st.text_area("Source resume text", load_raw_text(rec["source_file"]), height=420,
                     label_visibility="collapsed")


# ------------------------------------------------------------------------ main
def main() -> None:
    df, records = load_candidates()
    if df.empty:
        st.error("No parsed candidate data found. Run `python src/pipeline.py --seed-cache` first.")
        st.stop()

    st.title("Candidate Resume Search Platform")
    st.caption("Business Development talent sourcing - parsed resume search, screening and analytics")

    # ---------------- sidebar: requisition + filters
    sb = st.sidebar
    sb.markdown("## Requisition")

    # Requisitions are deep-linkable (?req=<n>) so a recruiter can send a colleague the
    # exact mandate view rather than a description of which filters to set.
    preset_keys = list(PRESETS.keys())
    try:
        default_idx = max(0, min(int(st.query_params.get("req", 0)), len(preset_keys) - 1))
    except (TypeError, ValueError):
        default_idx = 0

    preset_name = sb.selectbox("Preset mandate", preset_keys, index=default_idx)
    preset = PRESETS[preset_name]
    sb.caption(f"Shareable link for this mandate: `?req={preset_keys.index(preset_name)}`")

    # Filter options come from the canonical taxonomies first, with any additional
    # observed values appended. Driving options off the current pool alone would make
    # the interface change shape as data arrives, and would silently drop a filter the
    # moment nobody in the pool happens to match it.
    def options(canonical: list[str], observed) -> list[str]:
        seen = list(canonical)
        for value in sorted(set(observed)):
            if value and value not in seen:
                seen.append(value)
        return seen

    all_sectors = options(config.SECTORS, (s for row in df["sectors"] for s in row))
    all_regions = options(config.REGIONS, df["region"])
    all_strategies = options(config.STRATEGY_TYPES, df["strategy"])
    all_seniority = options(config.SENIORITY_LEVELS, df["seniority"])
    all_firm_types = options(config.FIRM_TYPES, df["firm_type"])
    all_sides = options(["Buy-Side", "Sell-Side", "Private Markets", "Corporate", "Academic"], df["market_side"])
    all_langs = sorted({l.split(" (")[0] for row in df["languages"] for l in row})

    def valid_defaults(values, allowed: list[str]) -> list[str]:
        """Presets must never crash the app if a taxonomy value is absent."""
        return [v for v in (values or []) if v in allowed]

    req_region = sb.selectbox(
        "Target region", ["Any"] + all_regions,
        index=(["Any"] + all_regions).index(preset["region"]) if preset.get("region") in all_regions else 0,
    )
    req_strategy = sb.selectbox(
        "Investment approach", ["Any"] + all_strategies,
        index=(["Any"] + all_strategies).index(preset["strategy"]) if preset.get("strategy") in all_strategies else 0,
    )
    req_sectors = sb.multiselect("Sector coverage", all_sectors,
                                 default=valid_defaults(preset.get("sectors"), all_sectors))
    sector_logic = sb.radio("Sector match", ["Any of these", "All of these"], horizontal=True)

    yr_lo, yr_hi = float(df["years"].min()), float(df["years"].max())
    req_years = sb.slider(
        "Target years of experience", 0.0, max(yr_hi + 2, 20.0),
        (float(preset.get("min_years", 0.0)), float(preset.get("max_years", max(yr_hi, 20.0)))), step=0.5,
    )
    # Experience bands on a requisition are guidance, not a hard boundary. Excluding a
    # candidate for being six months outside a band loses good people, so by default the
    # band drives the score (with linear decay outside it) and only becomes a hard cut if
    # the recruiter explicitly asks for one.
    enforce_years = sb.checkbox("Enforce experience band as a hard filter", value=False)

    with sb.expander("Additional filters", expanded=False):
        st.caption("The mandate's target seniority feeds the match score. Set a filter here only "
                   "if you want to exclude other levels outright.")
        f_seniority = st.multiselect("Seniority (hard filter)", all_seniority, default=[])
        f_firm = st.multiselect("Current firm type", all_firm_types)
        f_side = st.multiselect("Market side", all_sides)
        f_degree = st.multiselect("Highest degree", ["Doctorate / Medical", "Master's / MBA", "Bachelor's", "Unknown"])
        f_lang = st.multiselect("Language spoken", all_langs)
        f_cfa = st.checkbox("CFA charterholder only")
        f_md = st.checkbox("Medical degree only")
        f_employed = st.checkbox("Currently employed only")
        f_quality = st.slider("Minimum data-quality score", 0.0, 1.0, 0.0, 0.05)
        keyword = st.text_input("Keyword search", placeholder="e.g. long/short, backtesting, USFDA, Bloomberg")

    with sb.expander("Scoring weights", expanded=False):
        st.caption("The match score is a visible weighted sum. Adjust to reflect what the mandate really values.")
        weights = {k: st.slider(k.title(), 0, 50, v, 5) for k, v in DEFAULT_WEIGHTS.items()}

    # ---------------- vectorised filtering
    mask = pd.Series(True, index=df.index)
    if req_region != "Any":
        mask &= df["region"] == req_region
    if req_strategy != "Any":
        mask &= df["strategy"].isin([req_strategy, "Hybrid"])
    if req_sectors:
        if sector_logic == "All of these":
            mask &= df["sectors"].apply(lambda s: set(req_sectors).issubset(set(s)))
        else:
            mask &= df["sectors"].apply(lambda s: bool(set(req_sectors) & set(s)))
    if enforce_years:
        mask &= df["years"].between(req_years[0], req_years[1])
    if f_seniority:
        mask &= df["seniority"].isin(f_seniority)
    if f_firm:
        mask &= df["firm_type"].isin(f_firm)
    if f_side:
        mask &= df["market_side"].isin(f_side)
    if f_degree:
        mask &= df["degree_tier"].isin(f_degree)
    if f_lang:
        mask &= df["languages"].apply(lambda ls: any(l.split(" (")[0] in f_lang for l in ls))
    if f_cfa:
        mask &= df["has_cfa"]
    if f_md:
        mask &= df["has_md"]
    if f_employed:
        mask &= df["employed"]
    mask &= df["quality"] >= f_quality
    if keyword.strip():
        terms = [t.strip().lower() for t in keyword.split(",") if t.strip()]
        mask &= df["searchable"].apply(lambda txt: all(t in txt for t in terms))

    filtered = df[mask].copy()

    req = {
        "region": req_region, "strategy": req_strategy, "sectors": req_sectors,
        "min_years": req_years[0], "max_years": req_years[1],
        # Preset seniority informs scoring; an explicit filter selection overrides it.
        "seniority": f_seniority or valid_defaults(preset.get("seniority"), all_seniority),
    }
    # A match score is only meaningful once the requisition constrains something. With
    # every dimension set to "Any", every candidate scores near-perfectly, which is a
    # misleading ranking - so in browse mode we rank by experience and surface data
    # quality instead of a vacuous score.
    scoring_on = bool(req_sectors) or req_region != "Any" or req_strategy != "Any"

    if not filtered.empty:
        scored = filtered.apply(lambda r: score_candidate(r, req, weights), axis=1)
        filtered["match_score"] = [s for s, _ in scored]
        filtered["contributions"] = [c for _, c in scored]
        filtered = (filtered.sort_values(["match_score", "years"], ascending=[False, False])
                    if scoring_on else filtered.sort_values("years", ascending=False))

    # ---------------- header metrics
    m1, m2, m3, m4, m5 = st.columns(5)
    m1.metric("Candidates in pool", len(df))
    m2.metric("Matching filters", len(filtered))
    m3.metric("Mean experience", f"{filtered['years'].mean():.1f} yrs" if len(filtered) else "-")
    m4.metric("Top match", f"{filtered['match_score'].max():.0f}" if (len(filtered) and scoring_on) else "-")
    m5.metric("Records with flags", int((df["n_flags"] > 0).sum()))

    if preset_name != "-- No requisition (browse all) --":
        st.info(f"Scoring against preset mandate: **{preset_name}**", icon="\U0001F3AF")

    tab_results, tab_compare, tab_insights, tab_quality = st.tabs(
        ["Search results", "Compare candidates", "Talent pool insights", "Data quality"]
    )

    # ---------------- results
    with tab_results:
        if filtered.empty:
            st.warning("No candidates match these criteria. Try widening the experience band, "
                       "switching sector match to 'Any of these', or clearing the keyword search.")
        else:
            view = st.radio("View", ["Cards", "Table"], horizontal=True, label_visibility="collapsed")
            if view == "Table":
                cols = (["name", "match_score"] if scoring_on else ["name"]) + [
                    "region", "location", "current_employer", "current_title",
                    "strategy", "market_side", "seniority", "years", "coverage", "degree_tier",
                    "has_cfa", "quality", "n_flags",
                ]
                table = filtered[cols].rename(columns={
                    "name": "Candidate", "match_score": "Match", "region": "Region", "location": "Location",
                    "current_employer": "Current firm", "current_title": "Title", "strategy": "Approach",
                    "market_side": "Side", "seniority": "Seniority", "years": "Yrs",
                    "coverage": "Coverage", "degree_tier": "Degree", "has_cfa": "CFA",
                    "quality": "Quality", "n_flags": "Flags",
                })
                st.dataframe(table, use_container_width=True, hide_index=True)
            else:
                if not scoring_on:
                    st.caption("No requisition criteria set, so candidates are ranked by experience and "
                               "the right-hand figure shows data-quality score. Choose a preset mandate or "
                               "set a region, approach or sector to rank by match score.")
                for _, row in filtered.iterrows():
                    candidate_card(row, row["contributions"], scoring_on)

            st.download_button(
                "Download this shortlist (CSV)",
                filtered.drop(columns=["contributions", "searchable"]).to_csv(index=False).encode(),
                file_name="shortlist.csv", mime="text/csv",
            )

            st.divider()
            st.markdown("#### Candidate detail")
            names = filtered["name"].tolist()
            chosen = st.selectbox("Select a candidate to review", names, label_visibility="collapsed")
            row = filtered[filtered["name"] == chosen].iloc[0]
            candidate_detail(record_by_id(records, row["candidate_id"]),
                             row["contributions"] if scoring_on else None)

    # ---------------- comparison
    with tab_compare:
        st.markdown("#### Side-by-side comparison")
        st.caption("Compare shortlisted candidates on the dimensions a hiring manager asks about.")
        # Comparison is drawn from the whole pool, pre-populated with the current
        # shortlist. A recruiter frequently wants to benchmark a shortlisted candidate
        # against someone the filters excluded.
        shortlist = filtered["name"].tolist() if not filtered.empty else []
        picks = st.multiselect("Candidates", df["name"].tolist(),
                               default=shortlist[:3], max_selections=4)
        if len(picks) < 2:
            st.info("Select at least two candidates to compare.")
        else:
            src = filtered if not filtered.empty else df
            sub = src[src["name"].isin(picks)]
            missing = [p for p in picks if p not in sub["name"].tolist()]
            if missing:
                extra = df[df["name"].isin(missing)].copy()
                extra["match_score"] = float("nan")
                sub = pd.concat([sub, extra], ignore_index=True)
            comp = pd.DataFrame({
                r["name"]: {
                    "Match score": (f"{r['match_score']:.1f}" if pd.notna(r.get("match_score")) else "not scored"),
                    "Region": r["region"],
                    "Location": r["location"],
                    "Current firm": r["current_employer"],
                    "Title": r["current_title"],
                    "Approach": r["strategy"],
                    "Market side": r["market_side"],
                    "Firm type": r["firm_type"],
                    "Seniority": r["seniority"],
                    "Years experience": r["years"],
                    "Sectors": ", ".join(r["sectors"]),
                    "Max coverage universe": r["coverage"] or "n/a",
                    "Highest degree": r["highest_degree"] or r["degree_tier"],
                    "CFA": "Yes" if r["has_cfa"] else "No",
                    "Languages": ", ".join(r["languages"]) or "Not stated",
                    "Employers": r["n_employers"],
                    "Data quality": r["quality"],
                    "Flags": r["n_flags"],
                } for _, r in sub.iterrows()
            })
            st.dataframe(comp, use_container_width=True)

            radar_axes = ["Experience", "Sector breadth", "Coverage scale", "Credentials", "Data quality"]
            fig = go.Figure()
            for i, (_, r) in enumerate(sub.iterrows()):
                vals = [
                    min(r["years"] / 15, 1) * 100,
                    min(len(r["sectors"]) / 5, 1) * 100,
                    min((r["coverage"] or 0) / 75, 1) * 100,
                    (35 if r["degree_tier"] in {"Master's / MBA", "Doctorate / Medical"} else 15)
                    + (35 if r["has_cfa"] else 0) + (30 if r["has_md"] else 0),
                    r["quality"] * 100,
                ]
                fig.add_trace(go.Scatterpolar(r=vals + [vals[0]], theta=radar_axes + [radar_axes[0]],
                                              fill="toself", name=r["name"],
                                              line_color=PALETTE[i % len(PALETTE)], opacity=0.65))
            fig.update_layout(height=460, polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
                              margin=dict(t=30, b=10), legend=dict(orientation="h", y=-0.1))
            st.plotly_chart(fig, use_container_width=True)
            st.caption("Axes are normalised for comparability: experience against a 15-year scale, "
                       "sector breadth against 5 sectors, coverage against a 75-name universe.")

    # ---------------- insights
    with tab_insights:
        st.markdown("#### Talent pool composition")
        scope = st.radio("Scope", ["Full pool", "Filtered results"], horizontal=True)
        data = filtered if (scope == "Filtered results" and not filtered.empty) else df

        c1, c2 = st.columns(2)
        with c1:
            exploded = data.explode("sectors").dropna(subset=["sectors"])
            if not exploded.empty:
                pivot = (exploded.pivot_table(index="sectors", columns="region", values="candidate_id",
                                              aggfunc="count").fillna(0))
                fig = px.imshow(pivot, text_auto=True, color_continuous_scale="Blues", aspect="auto",
                                labels=dict(color="candidates"))
                fig.update_layout(title="Sector coverage by region", height=430,
                                  margin=dict(t=50, l=0, r=0, b=0), coloraxis_showscale=False)
                st.plotly_chart(fig, use_container_width=True)
        with c2:
            counts = data["strategy"].value_counts().reset_index()
            counts.columns = ["strategy", "n"]
            fig = px.pie(counts, names="strategy", values="n", hole=0.55,
                         color_discrete_sequence=PALETTE)
            fig.update_layout(title="Fundamental vs systematic mix", height=430, margin=dict(t=50))
            st.plotly_chart(fig, use_container_width=True)

        c3, c4 = st.columns(2)
        with c3:
            fig = px.histogram(data, x="years", nbins=10, color_discrete_sequence=[ACCENT])
            fig.update_layout(title="Experience distribution", xaxis_title="years of experience",
                              yaxis_title="candidates", height=380, margin=dict(t=50), bargap=0.08)
            st.plotly_chart(fig, use_container_width=True)
        with c4:
            fc = data["firm_type"].value_counts().reset_index()
            fc.columns = ["firm_type", "n"]
            fig = px.bar(fc, x="n", y="firm_type", orientation="h", color_discrete_sequence=[PRIMARY])
            fig.update_layout(title="Current firm type", xaxis_title="candidates", yaxis_title="",
                              height=380, margin=dict(t=50))
            st.plotly_chart(fig, use_container_width=True)

        c5, c6 = st.columns(2)
        with c5:
            scat = data.dropna(subset=["coverage"])
            if not scat.empty:
                fig = px.scatter(scat, x="years", y="coverage", color="region", hover_name="name",
                                 size=[14] * len(scat), color_discrete_sequence=PALETTE)
                fig.update_layout(title="Coverage universe vs experience", height=380,
                                  xaxis_title="years of experience", yaxis_title="names covered",
                                  margin=dict(t=50))
                st.plotly_chart(fig, use_container_width=True)
        with c6:
            skills = (data.explode("tools").dropna(subset=["tools"])["tools"]
                      .value_counts().head(12).reset_index())
            skills.columns = ["tool", "n"]
            if not skills.empty:
                fig = px.bar(skills, x="n", y="tool", orientation="h", color_discrete_sequence=[ACCENT])
                fig.update_layout(title="Most common tools and platforms", xaxis_title="candidates",
                                  yaxis_title="", height=380, margin=dict(t=50))
                st.plotly_chart(fig, use_container_width=True)

        st.markdown("##### Coverage gaps against the current requisition")
        if req_sectors:
            gap_rows = []
            for sector in req_sectors:
                in_region = df[(df["region"] == req_region) if req_region != "Any" else pd.Series(True, index=df.index)]
                gap_rows.append({
                    "Sector": sector,
                    "In pool": int(df["sectors"].apply(lambda s: sector in s).sum()),
                    "In target region": int(in_region["sectors"].apply(lambda s: sector in s).sum()),
                    "Matching full requisition": int(filtered["sectors"].apply(lambda s: sector in s).sum())
                    if not filtered.empty else 0,
                })
            st.dataframe(pd.DataFrame(gap_rows), use_container_width=True, hide_index=True)
            st.caption("Where 'matching full requisition' is zero or low, the pipeline is thin for that "
                       "sector in that market and sourcing effort should be redirected.")
        else:
            st.caption("Select one or more sectors in the requisition to see coverage gaps.")

    # ---------------- data quality
    with tab_quality:
        st.markdown("#### Extraction and data-quality review")
        st.caption("Every flag is raised by a documented rule or reported by the extraction model. "
                   "Nothing is auto-corrected, so a reviewer always sees the original discrepancy.")

        q1, q2, q3 = st.columns(3)
        q1.metric("Mean quality score", f"{df['quality'].mean():.2f}")
        q2.metric("Records with flags", int((df["n_flags"] > 0).sum()))
        q3.metric("Total flags", int(df["n_flags"].sum()))

        flag_rows = []
        for rec in records:
            for f in rec["data_quality_flags"]:
                flag_rows.append({
                    "Candidate": rec["full_name"], "Source": rec["source_file"],
                    "Origin": "extraction model" if f.startswith("[model]") else "validation rule",
                    "Finding": f.replace("[model] ", ""),
                })
            for g in rec.get("career_gaps", []):
                flag_rows.append({"Candidate": rec["full_name"], "Source": rec["source_file"],
                                  "Origin": "validation rule", "Finding": f"Career continuity: {g}"})
        fdf = pd.DataFrame(flag_rows)

        origin = st.multiselect("Filter by origin", sorted(fdf["Origin"].unique()),
                                default=sorted(fdf["Origin"].unique()))
        who = st.multiselect("Filter by candidate", sorted(fdf["Candidate"].unique()))
        shown = fdf[fdf["Origin"].isin(origin)]
        if who:
            shown = shown[shown["Candidate"].isin(who)]
        st.dataframe(shown, use_container_width=True, hide_index=True, height=420)

        fig = px.bar(df.sort_values("quality"), x="quality", y="name", orientation="h",
                     color="quality", color_continuous_scale="RdYlGn", range_color=[0, 1])
        fig.update_layout(title="Data-quality score by candidate", xaxis_title="quality score (1.0 = clean)",
                          yaxis_title="", height=430, margin=dict(t=50), coloraxis_showscale=False)
        st.plotly_chart(fig, use_container_width=True)

        st.download_button("Download data-quality report (CSV)", fdf.to_csv(index=False).encode(),
                           file_name="data_quality_report.csv", mime="text/csv")


if __name__ == "__main__":
    main()

Overwriting app.py


Launch it with:

```bash
streamlit run app.py
```

---
## 8. What the data says about this pool

Ten candidates is too small for statistical claims, but the composition is already
actionable, and these are the observations I would bring to a BD stakeholder.

**The pool is heavily fundamental and heavily healthcare.** Eight of ten are
fundamental investors; only Chen Li and Omar El-Hassan are systematic or quantitative.
Healthcare is the deepest sector with seven candidates, and three of the four
Asia-Pacific candidates are healthcare sell-side research analysts. If the open requisitions skew systematic or skew technology outside
the US, this pipeline does not cover them.

**Regional depth is uneven by sector.** US candidates cluster in TMT, consumer and
generalist mandates. APAC candidates cluster almost entirely in healthcare sell-side
research. Europe has two candidates covering entirely different things - one
fundamental equity generalist and one rates-and-credit quant developer. The
coverage-gap table in the app makes this concrete per requisition.

**Two profiles are genuinely differentiated.** Dr. Zara Al-Rashid pairs an MBBS with a
finance PGDM and ten years of pharma research - a clinical-depth profile that is hard
to source. Viktor Sharat pairs a biotechnology and bioinformatics M.Tech and a machine
learning publication with a decade of sell-side coverage. Both are sourcing assets a
keyword search on job titles would never surface, which is an argument for extracting
education and publications rather than just employment.

**One candidate is an obvious priority call.** Ryan Patel previously worked at
Millennium, is currently running a fundamental long/short book, and covers consumer,
TMT and healthcare. Prior familiarity with the platform is directly relevant context,
and the platform surfaces it because employers are extracted as structured fields.

**Two candidates are not currently placed.** Chen Li has no role recorded after
September 2023, and Viktor Sharat's document gives no dates at all. Both change how BD
would prioritise outreach, and neither is visible from a job title.

---
## 9. Designing for scale

The brief asks for a design that handles large volumes. The current implementation
already contains the parts that matter at scale; the rest is a described migration path
rather than speculation.

### Already built for it

| Concern | Implementation |
| --- | --- |
| Redundant LLM spend | Content-hash cache keyed on prompt version + document text; a re-run of an unchanged corpus makes zero API calls |
| Throughput | `ThreadPoolExecutor` with bounded concurrency, since provider rate limits rather than CPU are the constraint |
| Transient failure | Exponential backoff with jitter, plus a validation-repair round trip |
| Partial failure | One bad document cannot fail the batch; failures are collected and reported |
| Schema drift | Pydantic validation on every record, including cache reads, so a stale cache entry re-parses instead of breaking |
| UI responsiveness | `st.cache_data` on load, vectorised mask filtering, no per-row Python in the hot path |
| Reproducibility | Fixed `AS_OF_DATE`, temperature 0, versioned prompt |
| Regression safety | 25 unit tests over the validation and cleaning rules |

### Measured cost model

At roughly 1,100 input and 900 output tokens per resume (measured on this corpus), a
small frontier model at approximately $0.15 per million input and $0.60 per million
output tokens costs about **$0.0007 per resume**, or roughly **$700 per million
resumes** - and near zero for re-processing, because of the cache. Extraction cost is
not the binding constraint; document acquisition and review capacity are.

### Migration path from 10 to 1,000,000

**Ingestion.** Replace directory scanning with object storage plus an event queue (S3
plus SQS, or equivalent). Each message carries one document; workers scale
horizontally. Deduplicate on content hash before spending a single token - resume
corpora are full of near-duplicates from multiple agencies submitting the same person.

**Extraction.** Route by document complexity: a cheap model for clean single-column
resumes, a stronger model only for documents where confidence is low or validation
fails. Use provider batch APIs for the backlog, since a bulk backfill is not latency
sensitive and batch pricing is materially cheaper. Add OCR for scanned PDFs, which this
corpus does not contain but a real pipeline will hit within its first thousand files.

**Storage.** Move from JSON files to Postgres: a `candidates` table plus `roles`,
`education` and `flags` tables, with GIN indexes on the array columns for sector and
skill filtering, and a `tsvector` column for keyword search. Retain the raw text and
the model response for every record, because auditability is a hard requirement when
the output influences hiring.

**Search.** At a few hundred thousand records, structured filtering belongs in the
database rather than in pandas, and the app becomes a thin query layer over indexed
columns. Add embeddings over the highlight text for genuine semantic search - "someone
who has run a factor model on Asian equities" is a query no keyword index answers well,
and it is exactly how a BD user thinks. Store vectors in pgvector alongside the
structured record so a single query can combine both.

**Incremental processing.** Resumes are re-submitted with updates. Version records by
content hash, keep history, and diff versions so a recruiter can see what changed since
last contact.

**Operations.** Track extraction confidence, validation flag rates and per-field null
rates as monitored metrics. A jump in the null rate for `start_date` is how you find out
a new agency template broke the parser, before a recruiter finds out by getting bad
search results.

---
## 10. What I would build with more time

Ordered by value per hour of effort.

**1. An evaluation harness (highest priority).** The gap between this and a production
system is measurement. I would hand-label a gold-standard set of 30-50 resumes, then
report per-field precision and recall for every extracted field on every prompt or model
change. Without that, "the extraction is accurate" is an assertion. This is the first
thing I would build with another day, because it turns every subsequent change from a
guess into a measurement.

**2. Semantic search over experience.** Embed role highlights and support natural
language queries - "covered digital health from the buy side in Asia" - blended with
structured filters. This is the single largest usability gain for a BD user.

**3. Requisition-to-candidate matching from a job description.** Paste a real
requisition, have the model extract the structured criteria, and score the whole pool
against it automatically. That closes the loop from the actual artefact BD works from.

**4. Field-level confidence and targeted human review.** Confidence per field rather
than per document, with a review queue that surfaces only low-confidence fields. Review
capacity, not extraction, is the real bottleneck at volume.

**5. Entity resolution.** "J.P. Morgan", "J.P.Mogan", "JPMorgan Chase" and "J.P. Morgan
Asset Management" appear across this corpus and are partly the same institution, partly
not. A firm master table with aliases, plus firm tier and strategy metadata, would let
BD filter by pedigree rather than by string. Candidate-level deduplication belongs here
too, since the same person arrives from multiple agencies.

**6. Bias and fairness controls.** Any tool that ranks people needs this before it is
used in anger: excluding name, nationality, age proxies and gender-correlated signals
from scoring; auditing score distributions across demographic proxies; logging every
shortlist for review. The current scoring uses only sector, region, strategy, experience
and credentials, which is deliberate, but it has not been audited and I would not claim
otherwise.

**7. Access control, PII handling and audit logging.** Real resumes are personal data.
Encryption at rest, role-based access, retention policies aligned to GDPR and
equivalents, redaction of contact details until a recruiter has a legitimate reason to
see them, and an immutable log of who viewed and exported what.

**8. Pipeline hardening.** OCR for scanned documents, language detection and non-English
extraction, fixture resumes for each failure mode on top of the existing unit tests, and
CI that fails on a regression in extraction quality.

---

## 11. Honest limitations

Stated explicitly, because a reviewer will find them anyway and I would rather discuss
them directly.

* **No accuracy measurement.** Extraction correctness was verified by reading all 10
  resumes against the output, not by a labelled evaluation with reported metrics. On a
  corpus of ten that is feasible; it does not generalise, and item 1 above is the fix.
* **Ten records is not a talent pool.** The analytics in section 8 are directional
  observations, not statistics. Several charts would look different with a hundred
  records and should not be over-read.
* **Scoring weights are judgement, not evidence.** The default 35/20/20/15/10 split
  encodes my assumption that sector fit matters most for these mandates. It has not
  been validated against hiring outcomes, which is precisely why the weights are
  exposed as sliders rather than hard-coded and hidden.
* **Some classifications are genuinely arguable.** Chen Li does quantitative factor
  work but also fundamental valuation as an intern, and is classified systematic. The
  `sector_specialisation_detail` field preserves the verbatim evidence so a reviewer can
  disagree with the label without losing the underlying information.
* **Seniority titles are not comparable across firms.** "Associate" means different
  things at Goldman Sachs, Bain and Apollo. Seniority therefore informs the score but
  never gates a search by default.
* **The employer-contradiction check uses a curated firm list.** It reliably catches
  what is in this corpus and would need the entity-resolution work in item 5 to
  generalise.

## 12. How AI was used, and how the output was checked

The brief permits AI, so the honest account: an LLM performs the extraction step, which
is the point of the exercise, and AI assistance was used while writing this code.

What matters is the control structure around it. The model is constrained by a JSON
Schema at generation time and validated by Pydantic afterwards. It is forbidden from
doing arithmetic, so every derived number comes from deterministic Python. Its output
is checked against a dozen consistency rules that compare its extraction to itself and
to the source dates. It is required to report ambiguity rather than resolve it. The
extracted text it received is retained verbatim and is one click away in the app. And
every one of the ten source resumes was read end to end by hand and compared against
the extracted record - which is how the planted contradictions in section 6 were found,
and how the validation rules that catch them were designed.